# Run everything in one session

One notebook for the whole project, so the A100 only has to start once. It runs in order of cost, cheapest first, so if the session drops during the long LLM part, everything before it is already saved to Drive:

| Stage | What | GPU need | Time (A100, rough) |
|---|---|---|---|
| A. Data | fragment graphs for both modes, Table 1 statistics, fragment histogram, salts-by-label check | none | ~5 min |
| B. Models | fragment-GIN and atom-GIN, 3 seeds, both modes; base-model table | light | ~15–30 min the first time |
| C. Diagnostics | confusion matrices / parity plots, t-SNE & UMAP, readout attention | light | ~5 min |
| D. Explanations | Llama + GSPELL, text-only LLM, GNNExplainer, Grad-CAM, attention, random | **A100** | ~25 s per molecule per mode |
| E. Tables | side-by-side fidelity/size, agreement, counter-ion usage | none | seconds |
| E2. Faithfulness | exact fragment Shapley, removal counterfactuals, deletion curves, QA test with vs. without soft prompts | light; QA needs the A100 | ~1 s per molecule for Shapley; ~10 s per molecule for QA |
| F. Figures | explanation grids, fragment panels, heatmaps, soft-prompt UMAP | light | ~5 min |
| G. Export | all LaTeX rows in one file, a zip of figures and results | none | seconds |

**Everything is resumable.** Graphs, checkpoints and projectors that already exist are reused, and explained molecules are skipped. After a disconnect, just *Run all* again. Your earlier notebooks' outputs are picked up too.

**Before running:** *Runtime → A100*, a Colab secret `HF_TOKEN` with notebook access, then *Runtime → Run all*. With `N_EVAL = 30` the whole run is roughly 1.5–2.5 hours the first time; as far as I know, Colab Pro needs the browser tab to stay open for that long (background execution is a Pro+ feature).

## 1. Setup

In [ ]:
from google.colab import drive
drive.mount('/content/drive')
ROOT = "/content/drive/MyDrive/Jenifer code 2026"
OUT  = f"{ROOT}/dl_project"
CSVS = {"BBBP": f"{ROOT}/BBBP.csv", "BACE": f"{ROOT}/bace.csv", "ESOL": f"{ROOT}/esol.csv"}

!pip install -q rdkit torch_geometric bitsandbytes accelerate umap-learn
import os, sys, json, glob, time
os.makedirs(OUT, exist_ok=True); os.chdir(OUT); sys.path.insert(0, OUT)
import numpy as np, pandas as pd, torch
from IPython.display import display
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(DEVICE, torch.cuda.get_device_name(0) if torch.cuda.is_available() else "")
T0 = time.time()

## 2. Code (written into the project folder)

In [ ]:
%%writefile functional_groups.py
"""Functional-group and ring labels for fragments, plus list-free detection.

Three sources, combined in label_fragment():

1. Curated SMARTS (rule list). RDKit's FragmentDescriptors.csv (85 groups; the basis of
   RDKit's fr_* descriptors), minus entries that are metabolic-site heuristics or counts rather
   than functional groups, plus groups that list lacks (carbamate, sulfonic acid, sulfoxide,
   acyl halide, anhydride, alkene, alkyne, CF3, ...).
2. Ring names. Each ring of the fragment is matched to a table of common aromatic and
   aliphatic rings (pyridine, piperazine, ...); unknown rings are described by size/aromaticity.
3. Ertl's algorithm (J. Cheminform. 2017, 9:36) - no predefined list. Atoms are marked by
   general rules (heteroatoms; carbons in C=C, C#C or double/triple bonds to heteroatoms;
   acetal-type sp3 carbons; atoms of 3-membered heterocycles); connected marked atoms form one
   functional group, reported as a generalized SMILES type. Any group the curated list misses
   still gets a label this way ("group N=C(N)S" etc.).

Redundant labels are removed by subsumption: a label is dropped when every one of its matches
lies inside a strictly larger match of another label (so "carboxylic acid" absorbs "hydroxyl"
and "carbonyl" without hand-written exceptions).

mine_groups(smiles) counts Ertl types across a dataset: a data-driven vocabulary of the groups
that actually occur, with how often the curated list already names them.
"""

import csv
import os
from collections import Counter

from rdkit import Chem, RDConfig, RDLogger

RDLogger.DisableLog("rdApp.*")

LABEL_VERSION = "v2"   # bump when labels change; graphs/projectors/explanations are tagged with it

# ----------------------------------------------------------------------------- curated SMARTS

# RDKit entries that are not functional groups (metabolism sites, bicyclic flag, duplicates,
# overly broad counts) are skipped.
_SKIP = {"fr_C=O_noCOO", "fr_COO2", "fr_NH0", "fr_NH1", "fr_NH2", "fr_Ar_N", "fr_Ar_NH", "fr_Nhpyrrole",
         "fr_bicyclic", "fr_para_hydroxylation", "fr_allylic_oxid", "fr_aryl_methyl",
         "fr_Ndealkylation1", "fr_Ndealkylation2", "fr_HOCCN", "fr_ketone_Topliss",
         "fr_phenol_noOrthoHbond", "fr_Al_OH_noTert", "fr_nitro_arom_nonortho", "fr_ArN",
         "fr_unbrch_alkane", "fr_benzodiazepine", "fr_COO", "fr_C=O", "fr_halogen",
         "fr_imidazole", "fr_furan", "fr_thiophene", "fr_thiazole", "fr_oxazole", "fr_pyridine",
         "fr_piperdine", "fr_piperzine", "fr_morpholine", "fr_tetrazole", "fr_benzene"}   # rings: see RING_NAMES
# Shorter display names for RDKit entries
_RENAME = {"fr_Al_OH": "aliphatic hydroxyl", "fr_Ar_OH": "aromatic hydroxyl", "fr_Al_COO": "aliphatic carboxylic acid",
           "fr_Ar_COO": "aromatic carboxylic acid", "fr_quatN": "quaternary nitrogen", "fr_Imine": "imine",
           "fr_hdrzine": "hydrazine", "fr_hdrzone": "hydrazone", "fr_N-O": "hydroxylamine",
           "fr_priamide": "primary amide", "fr_guanido": "guanidine", "fr_isocyan": "isocyanate",
           "fr_isothiocyan": "isothiocyanate", "fr_sulfide": "thioether", "fr_SH": "thiol",
           "fr_C=S": "thiocarbonyl", "fr_sulfonamd": "sulfonamide", "fr_prisulfonamd": "primary sulfonamide",
           "fr_barbitur": "barbiturate", "fr_term_acetylene": "terminal alkyne", "fr_lactam": "beta-lactam",
           "fr_lactone": "lactone", "fr_epoxide": "epoxide", "fr_phos_acid": "phosphoric acid",
           "fr_phos_ester": "phosphate ester", "fr_nitro_arom": "nitroarene", "fr_dihydropyridine": "dihydropyridine",
           "fr_alkyl_carbamate": "alkyl carbamate", "fr_alkyl_halide": "alkyl halide", "fr_methoxy": "methoxy",
           "fr_aniline": "aniline (aryl amine)", "fr_ether": "ether", "fr_nitroso": "nitroso",
           "fr_amidine": "amidine", "fr_azide": "azide", "fr_azo": "azo", "fr_diazo": "diazo", "fr_imide": "imide",
           "fr_oxime": "oxime", "fr_sulfone": "sulfone", "fr_urea": "urea", "fr_nitro": "nitro"}
# Groups the RDKit list does not have
EXTRA_SMARTS = {
    "carboxylic acid": "[CX3](=O)[OX2H1]",
    "carboxylate": "[CX3](=O)[O-]",
    "carbamate": "[NX3][CX3](=O)[OX2][#6]",
    "carbonate": "[OX2][CX3](=O)[OX2]",
    "acyl halide": "[CX3](=O)[F,Cl,Br,I]",
    "anhydride": "[CX3](=O)[OX2][CX3]=O",
    "sulfonic acid": "[SX4](=O)(=O)[OX2H1,OX1-]",
    "sulfonate ester": "[SX4](=O)(=O)[OX2][#6]",
    "sulfoxide": "[$([SX3](=O)([#6])[#6]),$([SX3+]([O-])([#6])[#6])]",
    "sulfonyl": "[SX4](=O)(=O)",
    "disulfide": "[SX2][SX2]",
    "primary amine": "[NX3;H2;!$(N-C=[O,S,N]);!$(N-a);!$(N-S)][CX4]",
    "secondary amine": "[NX3;H1;!$(N-C=[O,S,N]);!$(N-a);!$(N-S)]([CX4])[CX4]",
    "tertiary amine": "[NX3;H0;!$(N-C=[O,S,N]);!$(N-a);!$(N-S);!$(N=*);!$(N#*)]([CX4])([CX4])[CX4]",
    "protonated amine (ammonium)": "[NX4+;!$(N~O)]",
    "enamine": "[NX3][CX3]=[CX3]",
    "enol ether": "[OX2]([#6])[CX3]=[CX3]",
    "alkene": "[CX3]=[CX3]",
    "alkyne": "[CX2]#[CX2]",
    "trifluoromethyl": "C(F)(F)F",
    "aryl halide": "c[F,Cl,Br,I]",
    "acetal / ketal": "[CX4]([OX2][#6])[OX2][#6]",
    "hemiacetal": "[CX4]([OX2H])[OX2][#6]",
    "boronic acid": "[BX3]([OX2H])[OX2H]",
    "N-oxide": "[#7+;!$([#7+](=O)[O-]);!$([#7+]=O)][OX1-]",
    "ring carbonyl (aromatic C=O)": "[#6;a]=[OX1]",
}


def _load_rdkit_groups():
    path = os.path.join(RDConfig.RDDataDir, "FragmentDescriptors.csv")
    out = {}
    if not os.path.exists(path):
        return out
    with open(path) as f:
        for row in csv.reader((l for l in f if l.strip() and not l.startswith("#")), delimiter="\t"):
            key, desc, smarts = row[0], row[1], row[2]
            if key in _SKIP:
                continue
            name = _RENAME.get(key) or desc.replace("Number of ", "").rstrip("s").strip().lower()
            out[name] = smarts
    return out


CURATED = {**_load_rdkit_groups(), **EXTRA_SMARTS}
_PATTERNS = {}
for _name, _s in CURATED.items():
    _p = Chem.MolFromSmarts(_s)
    if _p is not None:
        _PATTERNS[_name] = _p

# ----------------------------------------------------------------------------- rings

RING_NAMES = {
    # aromatic 6
    "c1ccccc1": "benzene ring", "c1ccncc1": "pyridine ring", "c1cncnc1": "pyrimidine ring",
    "c1cnccn1": "pyrazine ring", "c1ccnnc1": "pyridazine ring", "c1ncncn1": "triazine ring",
    "O=c1cccc[nH]1": "pyridone ring",
    # aromatic 5
    "c1cc[nH]c1": "pyrrole ring", "c1ccoc1": "furan ring", "c1ccsc1": "thiophene ring",
    "c1c[nH]cn1": "imidazole ring", "c1cn[nH]c1": "pyrazole ring", "c1cocn1": "oxazole ring",
    "c1cnoc1": "isoxazole ring", "c1cscn1": "thiazole ring", "c1cnsc1": "isothiazole ring",
    "c1nc[nH]n1": "1,2,4-triazole ring", "c1cn[nH]n1": "1,2,3-triazole ring", "c1nn[nH]n1": "tetrazole ring",
    "c1nnco1": "1,3,4-oxadiazole ring", "c1ncon1": "1,2,4-oxadiazole ring", "c1nncs1": "thiadiazole ring",
    # aliphatic
    "C1CC1": "cyclopropane", "C1CCC1": "cyclobutane", "C1CCCC1": "cyclopentane", "C1CCCCC1": "cyclohexane",
    "C1CCCCCC1": "cycloheptane", "C1CCNC1": "pyrrolidine ring", "C1CCNCC1": "piperidine ring",
    "C1CNCCN1": "piperazine ring", "C1COCCN1": "morpholine ring", "C1CCOC1": "tetrahydrofuran ring",
    "C1CCOCC1": "tetrahydropyran ring", "C1CSCN1": "thiazolidine ring", "C1CO1": "oxirane ring",
    "C1CN1": "aziridine ring", "C1COCO1": "dioxolane ring", "C1CCSC1": "tetrahydrothiophene ring",
    "C1=CCCCC1": "cyclohexene", "C1CNC1": "azetidine ring", "C1=CCCC1": "cyclopentene",
    "C1=CNC=CC1": "1,4-dihydropyridine ring", "C1=CCNCC1": "tetrahydropyridine ring",
    "O=C1CCCN1": "pyrrolidinone (lactam) ring", "C1CCNCC1": "piperidine ring",
}


def _canon(smi):
    m = Chem.MolFromSmiles(smi)
    return Chem.MolToSmiles(m) if m is not None else smi


RING_NAMES = {_canon(k): v for k, v in RING_NAMES.items()}   # keys in RDKit canonical form


def _ring_key(mol, ring):
    """Ring atoms only, all exocyclic substituents removed, charges and H counts reset."""
    em = Chem.RWMol()
    idx = {}
    for a in ring:
        at = mol.GetAtomWithIdx(a)
        n = Chem.Atom(at.GetAtomicNum())
        n.SetIsAromatic(at.GetIsAromatic())
        idx[a] = em.AddAtom(n)
    for i, a in enumerate(ring):
        b = ring[(i + 1) % len(ring)]
        bond = mol.GetBondBetweenAtoms(a, b)
        em.AddBond(idx[a], idx[b], bond.GetBondType())
    m = em.GetMol()
    try:
        m.UpdatePropertyCache(strict=False)
        Chem.SanitizeMol(m, Chem.SanitizeFlags.SANITIZE_ALL ^ Chem.SanitizeFlags.SANITIZE_PROPERTIES, catchErrors=True)
        smi = Chem.MolToSmiles(m)
        # aromatic N with implicit H: try the [nH] form so pyrrole/imidazole-type rings match the table
        if smi not in RING_NAMES:
            for alt in (smi.replace("n", "[nH]", 1),):
                mm = Chem.MolFromSmiles(alt)
                if mm is not None and Chem.MolToSmiles(mm) in RING_NAMES:
                    return Chem.MolToSmiles(mm)
        return smi
    except Exception:
        return Chem.MolFragmentToSmiles(mol, atomsToUse=list(ring))


def ring_labels(mol, atoms):
    atoms = set(atoms)
    ri = mol.GetRingInfo()
    rings = [r for r in ri.AtomRings() if set(r) <= atoms]
    labels = []
    for r in rings:
        key = _ring_key(mol, r)
        if key in RING_NAMES:
            labels.append(RING_NAMES[key])
        else:
            arom = all(mol.GetAtomWithIdx(a).GetIsAromatic() for a in r)
            het = sorted({mol.GetAtomWithIdx(a).GetSymbol() for a in r} - {"C"})
            labels.append(f"{len(r)}-membered {'aromatic' if arom else 'aliphatic'} "
                          + (f"{'/'.join(het)}-heterocycle" if het else "carbocycle"))
    fused = any(len(set(r1) & set(r2)) >= 2 for i, r1 in enumerate(rings) for r2 in rings[i + 1:])
    if fused:
        labels.append("fused ring system")
    return labels


# ----------------------------------------------------------------------------- Ertl IFG

def ertl_groups(mol):
    """Ertl's functional-group identification. Returns a list of (atom set, type SMILES)."""
    marked = set()
    for at in mol.GetAtoms():
        if at.GetAtomicNum() not in (1, 6):                       # heteroatoms
            marked.add(at.GetIdx())
    for b in mol.GetBonds():                                      # C=C, C#C, C=X, C#X (non-aromatic)
        if b.GetBondType() in (Chem.BondType.DOUBLE, Chem.BondType.TRIPLE) and not b.GetIsAromatic():
            marked.update((b.GetBeginAtomIdx(), b.GetEndAtomIdx()))
    for at in mol.GetAtoms():                                     # acetal-type sp3 carbons
        if at.GetAtomicNum() == 6 and at.GetHybridization() == Chem.HybridizationType.SP3:
            het = [n for n in at.GetNeighbors() if n.GetAtomicNum() in (7, 8, 16)
                   and all(bb.GetBondType() == Chem.BondType.SINGLE for bb in n.GetBonds())]
            if len(het) >= 2:
                marked.add(at.GetIdx())
    for r in mol.GetRingInfo().AtomRings():                       # oxirane / aziridine / thiirane
        if len(r) == 3 and any(mol.GetAtomWithIdx(a).GetAtomicNum() in (7, 8, 16) for a in r):
            marked.update(r)
    # connected components of marked atoms
    groups, seen = [], set()
    for a in marked:
        if a in seen:
            continue
        stack, comp = [a], set()
        while stack:
            x = stack.pop()
            if x in comp:
                continue
            comp.add(x)
            stack += [n.GetIdx() for n in mol.GetAtomWithIdx(x).GetNeighbors() if n.GetIdx() in marked]
        seen |= comp
        groups.append(comp)
    out = []
    for comp in groups:
        try:
            t = Chem.MolFragmentToSmiles(mol, atomsToUse=sorted(comp), canonical=True)
        except Exception:
            t = "?"
        out.append((comp, t))
    return out


# ----------------------------------------------------------------------------- labelling

def _subsume(matches):
    """matches: {label: [atom sets]}. Drop a label when every one of its matches lies inside a
    larger match of another label, or coincides with a match of a more specific label
    (longer SMARTS), e.g. 'sulfonamide' gives way to 'primary sulfonamide'."""
    spec = {l: len(CURATED.get(l, "")) for l in matches}
    keep = []
    for lab, ms in matches.items():
        if "$(" in CURATED.get(lab, ""):      # recursive SMARTS match only an anchor atom: keep
            keep.append(lab)
            continue
        def absorbed(m):
            for l2, ms2 in matches.items():
                if l2 == lab:
                    continue
                for o in ms2:
                    if m < o or (m == o and spec[l2] > spec[lab]):
                        return True
            return False
        if all(absorbed(m) for m in ms):
            continue
        keep.append(lab)
    return keep


ELEMENT_NAMES = {"N": "nitrogen", "O": "oxygen", "S": "sulfur", "C": "carbon", "P": "phosphorus"}


def label_fragment(mol, atoms, max_labels=6):
    """Readable labels for the fragment made of `atoms` (indices into mol):
      - single-atom fragments: '<element> linker atom'
      - rings in the fragment (named when known)
      - curated groups fully inside the fragment
      - 'part of <group>' for curated groups that BRICS cut through (e.g. an amide split
        between its C=O and N fragments)
      - Ertl groups (list-free) for anything the curated list did not cover"""
    atoms = set(atoms)
    inside, crossing = {}, {}
    for name, patt in _PATTERNS.items():
        for m in mol.GetSubstructMatches(patt):
            m = set(m)
            if m <= atoms:
                inside.setdefault(name, []).append(m)
            elif m & atoms:
                crossing.setdefault(name, []).append(m)
    fg_labels = _subsume(inside)
    marked = set().union(*(c for c, _ in ertl_groups(mol))) if mol.GetNumAtoms() else set()
    inside_sets = [m for l in fg_labels for m in inside[l]]
    # 'part of X' only when the fragment holds part of X's functional core (a heteroatom or a
    # multiply bonded carbon), not just a carbon it is attached to, and nothing inside already covers it
    crossing = {l: [m for m in ms if (m & atoms & marked) and not any((m & atoms) <= im for im in inside_sets)]
                for l, ms in crossing.items()}
    crossing = {l: ms for l, ms in crossing.items() if ms}
    part_labels = [f"part of {l}" for l in _subsume(crossing) if l not in fg_labels]
    named_atoms = set()
    for l in fg_labels:
        for m in inside[l]:
            named_atoms |= m
    for l in crossing:
        for m in crossing[l]:
            named_atoms |= (m & atoms)
    ring_l = ring_labels(mol, atoms)
    ring_atoms = {a for r in mol.GetRingInfo().AtomRings() if set(r) <= atoms for a in r}
    extra = []
    for comp, t in ertl_groups(mol):
        local = comp & atoms
        if not local or local & named_atoms:
            continue
        if local <= ring_atoms and all(mol.GetAtomWithIdx(a).GetIsAromatic() for a in local):
            continue      # aromatic ring heteroatoms: already described by the ring name
        if len(local) == 1 and mol.GetAtomWithIdx(next(iter(local))).GetSymbol() in ("F", "Cl", "Br", "I"):
            extra.append("halogen")
            continue
        if comp <= atoms:
            extra.append(f"group {t}")
        else:
            part = Chem.MolFragmentToSmiles(mol, atomsToUse=sorted(local), canonical=True)
            extra.append(f"part of group {t}" if len(t) <= 12 else f"group {part} (cut)")
    head = []
    if len(atoms) == 1:
        sym = mol.GetAtomWithIdx(next(iter(atoms))).GetSymbol()
        head = [f"{ELEMENT_NAMES.get(sym, sym)} linker atom"]
    fg_labels = [l for l in fg_labels if not any(l in r for r in ring_l)]   # e.g. 'dihydropyridine'
    labels = list(dict.fromkeys(head + ring_l + fg_labels + part_labels + extra))
    if not labels and all(mol.GetAtomWithIdx(a).GetSymbol() == "C" for a in atoms):
        labels = ["alkyl chain"]
    return labels[:max_labels]


def mine_groups(smiles_list, top=40):
    """Data-driven vocabulary: Ertl group types found in a set of molecules, how many molecules
    contain each, and whether the curated list names them. Returns a pandas DataFrame."""
    import pandas as pd
    counts, named = Counter(), Counter()
    for smi in smiles_list:
        mol = Chem.MolFromSmiles(smi)
        if mol is None:
            continue
        seen = set()
        cover = set(a for p in _PATTERNS.values() for m in mol.GetSubstructMatches(p) for a in m)
        named_rings = set(a for r in mol.GetRingInfo().AtomRings() if _ring_key(mol, r) in RING_NAMES for a in r)
        for comp, t in ertl_groups(mol):
            if t in seen:
                continue
            seen.add(t)
            counts[t] += 1
            # same rule as label_fragment: overlapping a curated match, or sitting in a named ring
            if comp & cover or comp <= named_rings:
                named[t] += 1
    rows = [{"ertl_type": t, "molecules": c, "pct_named_by_curated": 100 * named[t] / c}
            for t, c in counts.most_common(top)]
    return pd.DataFrame(rows)

In [ ]:
%%writefile fragment_graphs.py
"""
BRICS fragment graphs for MoleculeNet (BBBP, BACE, ESOL), scaffold splits.

Each molecule becomes a graph whose NODES are BRICS fragments and whose EDGES are
the bonds BRICS broke. Every node carries text (fragment SMILES + functional-group
names), so each molecule is a small text-attributed graph that GSPELL can explain.

Two modes
    default   counter-ions and other extra components are removed (largest component kept)
    --ionic   BRICS + ionic decomposition: every component is kept; metal-heteroatom bonds
              written as covalent (e.g. CC(=O)O[Na]) are split into ions; each counter-ion /
              salt partner is one node, linked to the main molecule by an "ionic" edge

Usage
    python fragment_graphs.py --demo "O=C(Nc1ccccc1)c1ccccc1"   # inspect one molecule
    python fragment_graphs.py --demo "CCN(CC)CC.Cl" --ionic     # same, keeping the salt
    python fragment_graphs.py --stats                           # dataset statistics table (CPU, minutes)
    python fragment_graphs.py --build                           # also save PyG graphs to ./frag_graphs/

Requires: rdkit, numpy, pandas, deepchem (for scaffold splits); torch + torch_geometric only for --build.
"""

import argparse
import os

import numpy as np
import pandas as pd
from rdkit import Chem, RDLogger
from rdkit.Chem import BRICS, rdFingerprintGenerator, rdMolDescriptors
from rdkit.Chem.MolStandardize import rdMolStandardize

RDLogger.DisableLog("rdApp.*")

DATASETS = {
    # name: (DeepChem loader, task type)
    "BBBP": ("load_bbbp", "classification"),
    "BACE": ("load_bace_classification", "classification"),
    "ESOL": ("load_delaney", "regression"),
}
MAX_FRAGMENTS_FOR_EXACT_SHAPLEY = 12
FP_BITS = 512

# Readable functional-group names for node text. A group is assigned to a fragment
# only if every atom of the match lies inside that fragment.
FUNCTIONAL_GROUPS = {
    "carboxylic acid": "[CX3](=O)[OX2H1]",
    "carboxylate": "[CX3](=O)[O-]",
    "protonated amine (ammonium)": "[NX4+;!$(N~O)]",
    "ester": "[#6][CX3](=O)[OX2H0][#6]",
    "amide": "[NX3][CX3]=[OX1]",
    "ketone or aldehyde": "[#6][CX3](=O)[#1,#6]",
    "hydroxyl": "[OX2H][#6]",
    "ether": "[OD2]([#6])[#6]",
    "primary amine": "[NX3;H2;!$(NC=O)][#6]",
    "secondary amine": "[NX3;H1;!$(NC=O)]([#6])[#6]",
    "tertiary amine": "[NX3;H0;!$(NC=O);!$(N=*);!a]([#6])([#6])[#6]",
    "nitro": "[N+](=O)[O-]",
    "nitrile": "C#N",
    "sulfonamide": "S(=O)(=O)N",
    "sulfone or sulfonyl": "[SX4](=O)(=O)",
    "thioether": "[SX2]([#6])[#6]",
    "trifluoromethyl": "C(F)(F)F",
    "halogen": "[F,Cl,Br,I]",
    "phosphate or phosphonate": "P(=O)(O)",
    "benzene ring": "c1ccccc1",
    "aromatic nitrogen heterocycle": "[n]",
    "aromatic oxygen or sulfur heterocycle": "[o,s]",
    "alkene": "[CX3]=[CX3]",
    "carbonyl": "[CX3]=[OX1]",
}
# Drop generic labels when a more specific one already covers them.
SUPPRESS = {
    "carboxylic acid": {"hydroxyl", "carbonyl", "ketone or aldehyde"},
    "carboxylate": {"carbonyl", "ketone or aldehyde"},
    "ester": {"ether", "carbonyl", "ketone or aldehyde"},
    "amide": {"carbonyl", "ketone or aldehyde"},
    "ketone or aldehyde": {"carbonyl"},
    "sulfonamide": {"sulfone or sulfonyl"},
    "trifluoromethyl": {"halogen"},
}
ELEMENT_NAMES = {"N": "nitrogen", "O": "oxygen", "S": "sulfur", "C": "carbon", "P": "phosphorus"}
_FG_PATTERNS = {name: Chem.MolFromSmarts(s) for name, s in FUNCTIONAL_GROUPS.items()}
_MORGAN = rdFingerprintGenerator.GetMorganGenerator(radius=2, fpSize=FP_BITS)
_LARGEST = rdMolStandardize.LargestFragmentChooser()
_METALS = rdMolStandardize.MetalDisconnector()

# Readable names for common counter-ions and salt partners (canonical SMILES of the component).
ION_NAMES = {
    "[Cl-]": "chloride ion", "Cl": "chloride (HCl salt)", "[Br-]": "bromide ion", "Br": "bromide (HBr salt)",
    "[I-]": "iodide ion", "I": "iodide (HI salt)", "[F-]": "fluoride ion",
    "[Na+]": "sodium ion", "[K+]": "potassium ion", "[Li+]": "lithium ion", "[Ca+2]": "calcium ion",
    "[Mg+2]": "magnesium ion", "[Zn+2]": "zinc ion", "[NH4+]": "ammonium ion", "N": "ammonia",
    "O": "water (hydrate)",
    "O=S(=O)(O)O": "sulfuric acid (sulfate salt)", "O=S(=O)([O-])[O-]": "sulfate ion",
    "O=S(=O)([O-])O": "hydrogen sulfate ion",
    "CS(=O)(=O)O": "methanesulfonic acid (mesylate salt)", "CS(=O)(=O)[O-]": "mesylate ion",
    "Cc1ccc(S(=O)(=O)O)cc1": "p-toluenesulfonic acid (tosylate salt)",
    "O=P(O)(O)O": "phosphoric acid (phosphate salt)", "O=[N+]([O-])O": "nitric acid (nitrate salt)",
    "O=C(O)/C=C\\C(=O)O": "maleic acid (maleate salt)", "O=C(O)/C=C/C(=O)O": "fumaric acid (fumarate salt)",
    "O=C(O)C(=O)O": "oxalic acid (oxalate salt)", "O=C(O)CC(O)(CC(=O)O)C(=O)O": "citric acid (citrate salt)",
    "O=C(O)C(O)C(O)C(=O)O": "tartaric acid (tartrate salt)", "CC(=O)O": "acetic acid (acetate salt)",
}
# Basic nitrogens that an acid counter-ion would protonate (amines and aromatic N, not amides)
_BASIC_N = Chem.MolFromSmarts("[NX3;!$(N-C=[O,S,N]);!$(N-S(=O)=O);!$(N-a)]")
_AROM_N = Chem.MolFromSmarts("[nX2]")
BOND_TYPES = [Chem.BondType.SINGLE, Chem.BondType.DOUBLE, Chem.BondType.TRIPLE, Chem.BondType.AROMATIC]


# ----------------------------------------------------------------------------- fragmentation

def prepare_mol(smiles, ionic=False):
    """Parse SMILES. Default: keep the largest component (drops counter-ions such as '.Cl').
    ionic=True: keep every component and split metal-heteroatom bonds into ions
    (RDKit MetalDisconnector; atom order is preserved).
    Returns (mol, changed) or (None, False) if the SMILES is invalid."""
    mol = Chem.MolFromSmiles(smiles)
    if mol is None:
        return None, False
    if ionic:
        n_before = len(Chem.GetMolFrags(mol))
        try:
            split = _METALS.Disconnect(mol)
            Chem.SanitizeMol(split)       # recompute rings/valences after bonds were removed
            mol = split
        except Exception:                 # keep the molecule as written rather than drop it,
            pass                          # so both modes contain the same molecules
        return mol, len(Chem.GetMolFrags(mol)) != n_before
    if len(Chem.GetMolFrags(mol)) > 1:
        return _LARGEST.choose(mol), True
    return mol, False


def brics_partition(mol, allowed_atoms=None):
    """Split a molecule at BRICS bonds, keeping atom indices.

    Unlike BRICS.BRICSDecompose (which returns a set of unique SMILES), this keeps
    duplicate fragments, maps every atom to its fragment, and records which bonds
    were broken, which is what you need to build fragment-level edges.

    Returns
        groups:       list of tuples of atom indices, one per fragment
        frag_of_atom: list, fragment index for each atom
        broken:       list of (atom_i, atom_j) for each broken bond
    """
    pairs = sorted({tuple(sorted(p)) for p, _ in BRICS.FindBRICSBonds(mol)})
    if allowed_atoms is not None:           # ionic mode: counter-ions stay whole
        pairs = [p for p in pairs if p[0] in allowed_atoms and p[1] in allowed_atoms]
    bond_ids = [mol.GetBondBetweenAtoms(i, j).GetIdx() for i, j in pairs]
    pieces = Chem.FragmentOnBonds(mol, bond_ids, addDummies=False) if bond_ids else mol
    groups = list(Chem.GetMolFrags(pieces, asMols=False, sanitizeFrags=False))
    frag_of_atom = [None] * mol.GetNumAtoms()
    for f, group in enumerate(groups):
        for a in group:
            frag_of_atom[a] = f
    return groups, frag_of_atom, pairs


LABEL_SCHEME = "extended"   # "extended": functional_groups.label_fragment (curated + rings + Ertl)
                            # "basic": the original short SMARTS list below


def label_version():
    if LABEL_SCHEME == "extended":
        from functional_groups import LABEL_VERSION
        return LABEL_VERSION
    return "v1"


def functional_groups(mol, group):
    """Readable labels for one fragment, matched in the parent molecule."""
    if LABEL_SCHEME == "extended":
        from functional_groups import label_fragment
        return label_fragment(mol, group) or ["other"]
    atoms = set(group)
    labels = [name for name, patt in _FG_PATTERNS.items()
              if any(set(m) <= atoms for m in mol.GetSubstructMatches(patt))]
    drop = set().union(*(SUPPRESS.get(l, set()) for l in labels))
    labels = [l for l in labels if l not in drop]
    if labels:
        return labels
    if len(group) == 1:   # single-atom linker left between two BRICS cuts
        sym = mol.GetAtomWithIdx(group[0]).GetSymbol()
        return [f"{ELEMENT_NAMES.get(sym, sym)} linker atom"]
    if any(mol.GetAtomWithIdx(a).GetIsAromatic() for a in group):
        return ["aromatic ring"]
    if any(mol.GetAtomWithIdx(a).IsInRing() for a in group):
        return ["aliphatic ring"]
    if all(mol.GetAtomWithIdx(a).GetSymbol() == "C" for a in group):
        return ["alkyl chain"]
    return ["other"]


def fragment_features(mol, group, logp_contribs, mr_contribs, tpsa_contribs):
    """Morgan fingerprint of the fragment (centered on its atoms) + summed atom-level
    descriptor contributions computed on the parent molecule."""
    fp = _MORGAN.GetFingerprintAsNumPy(mol, fromAtoms=list(group)).astype(np.float32)
    atoms = [mol.GetAtomWithIdx(a) for a in group]
    desc = np.array([
        len(group),                                              # heavy atoms
        sum(a.GetIsAromatic() for a in atoms) / len(group),      # aromatic fraction
        sum(a.IsInRing() for a in atoms) / len(group),           # ring-atom fraction
        sum(a.GetSymbol() == "N" for a in atoms),
        sum(a.GetSymbol() == "O" for a in atoms),
        sum(a.GetSymbol() == "S" for a in atoms),
        sum(a.GetSymbol() in ("F", "Cl", "Br", "I") for a in atoms),
        sum(a.GetTotalNumHs() for a in atoms),
        sum(logp_contribs[a] for a in group),                    # Crippen logP contribution
        sum(mr_contribs[a] for a in group),                      # Crippen molar refractivity
        sum(tpsa_contribs[a] for a in group),                    # polar surface area contribution
    ], dtype=np.float32)
    return np.concatenate([fp, desc])


def bond_features(bond, ionic_mode=False):
    f = [float(bond.GetBondType() == t) for t in BOND_TYPES] + [float(bond.GetIsConjugated())]
    return f + [0.0] if ionic_mode else f


IONIC_EDGE = [0.0, 0.0, 0.0, 0.0, 0.0, 1.0]


def main_component(mol, components):
    """The parent molecule: the largest component that is not a known counter-ion or salt
    partner (so e.g. tosylic acid never wins over the drug it is paired with)."""
    def known(c):
        return Chem.MolFragmentToSmiles(mol, atomsToUse=list(c), canonical=True) in ION_NAMES
    cands = [i for i, c in enumerate(components) if not known(c)] or list(range(len(components)))
    return max(cands, key=lambda i: (len(components[i]),
                                     sum(mol.GetAtomWithIdx(a).GetSymbol() == "C" for a in components[i]), -i))


def ion_name(mol, comp_atoms):
    smi = Chem.MolFragmentToSmiles(mol, atomsToUse=list(comp_atoms), canonical=True)
    if smi in ION_NAMES:
        return ION_NAMES[smi]
    charge = sum(mol.GetAtomWithIdx(a).GetFormalCharge() for a in comp_atoms)
    return "cation" if charge > 0 else "anion" if charge < 0 else "salt partner / co-crystal component"


def ionic_links(mol, components, main, frag_of_atom):
    """Rules for linking each non-main component (counter-ion, salt partner, solvent) to the
    rest of the molecule, since no covalent bond connects them:
      1. charged component -> fragments holding atoms of opposite formal charge
         (prefer the main component; otherwise any other component)
      2. neutral component (e.g. 'Cl' for an HCl salt, an acid partner, water) ->
         the main-component fragment holding the most basic nitrogen (amine first,
         then aromatic N); if there is none, the main component's largest fragment
    Returns a set of (fragment_u, fragment_v) pairs."""
    links = set()
    main_atoms = set(components[main])

    def frag_sizes():
        sizes = {}
        for a in main_atoms:
            sizes[frag_of_atom[a]] = sizes.get(frag_of_atom[a], 0) + 1
        return sizes

    for ci, comp in enumerate(components):
        if ci == main:
            continue
        src = frag_of_atom[comp[0]]            # non-main components are single nodes
        charge = sum(mol.GetAtomWithIdx(a).GetFormalCharge() for a in comp)
        targets = []
        if charge != 0:
            sign = 1 if charge > 0 else -1
            opposite = [a.GetIdx() for a in mol.GetAtoms()
                        if a.GetFormalCharge() * sign < 0 and a.GetIdx() not in comp]
            in_main = [a for a in opposite if a in main_atoms]
            targets = sorted({frag_of_atom[a] for a in (in_main or opposite)})
        if not targets:
            for patt in (_BASIC_N, _AROM_N):
                hits = [m[0] for m in mol.GetSubstructMatches(patt) if m[0] in main_atoms]
                if hits:
                    targets = [frag_of_atom[hits[0]]]
                    break
        if not targets:
            sizes = frag_sizes()
            targets = [max(sizes, key=sizes.get)]
        for t in targets:
            if t != src:
                links.add((src, t))
    return links


def fragment_graph(smiles, ionic=False):
    """Everything needed to build one fragment graph, as plain Python/numpy.
    Returns None for invalid SMILES. See the module docstring for the two modes."""
    mol, changed = prepare_mol(smiles, ionic)
    if mol is None:
        return None

    components = [list(c) for c in Chem.GetMolFrags(mol)]
    main = main_component(mol, components)
    allowed = set(components[main]) if ionic else None
    groups, frag_of_atom, broken = brics_partition(mol, allowed)
    comp_of_atom = {a: ci for ci, c in enumerate(components) for a in c}

    crippen = rdMolDescriptors._CalcCrippenContribs(mol)
    logp = [c[0] for c in crippen]
    mr = [c[1] for c in crippen]
    tpsa = list(rdMolDescriptors._CalcTPSAContribs(mol))

    texts, labels, counterion = [], [], []
    for group in groups:
        frag_smi = Chem.MolFragmentToSmiles(mol, atomsToUse=list(group), canonical=True)
        is_ci = ionic and comp_of_atom[group[0]] != main
        if is_ci:
            fg = [ion_name(mol, group)]
            texts.append(f"{frag_smi} (counter-ion: {fg[0]})")
        else:
            fg = functional_groups(mol, group)
            texts.append(f"{frag_smi} ({', '.join(fg)})")
        labels.append(fg)
        counterion.append(is_ci)

    x = np.stack([fragment_features(mol, g, logp, mr, tpsa) for g in groups])
    if ionic:   # net formal charge of the fragment, and whether it is a counter-ion node
        extra = np.array([[sum(mol.GetAtomWithIdx(a).GetFormalCharge() for a in g), float(ci)]
                          for g, ci in zip(groups, counterion)], dtype=np.float32)
        x = np.concatenate([x, extra], axis=1)

    edges, edge_attr = [], []
    for i, j in broken:
        u, v = frag_of_atom[i], frag_of_atom[j]
        feats = bond_features(mol.GetBondBetweenAtoms(i, j), ionic)
        edges += [(u, v), (v, u)]
        edge_attr += [feats, feats]
    n_ionic = 0
    if ionic and len(components) > 1:
        for u, v in sorted(ionic_links(mol, components, main, frag_of_atom)):
            edges += [(u, v), (v, u)]
            edge_attr += [IONIC_EDGE, IONIC_EDGE]
            n_ionic += 1

    return {
        "smiles": Chem.MolToSmiles(mol),
        "ionic": ionic,
        "salt_stripped": changed and not ionic,
        "metal_disconnected": changed and ionic,
        "n_components": len(components),
        "n_counterion_nodes": sum(counterion),
        "n_ionic_edges": n_ionic,
        "num_fragments": len(groups),
        "fragment_text": texts,
        "fragment_labels": labels,
        "counterion": counterion,
        "atom_groups": [list(g) for g in groups],
        "frag_of_atom": frag_of_atom,
        "x": x,
        "edges": edges,
        "edge_attr": edge_attr,
    }


# ----------------------------------------------------------------------------- data

CSV_COLUMNS = {
    # name: (SMILES column, label column) in the standard MoleculeNet CSVs
    "BBBP": ("smiles", "p_np"),
    "BACE": ("mol", "Class"),
    "ESOL": ("smiles", "measured log solubility in mols per litre"),
}


def scaffold_split(smiles_list, frac_train=0.8, frac_valid=0.1):
    """Deterministic Bemis-Murcko scaffold split (same procedure as DeepChem's
    ScaffoldSplitter): group by scaffold, largest groups go to train first."""
    from rdkit.Chem.Scaffolds import MurckoScaffold
    groups = {}
    for i, smi in enumerate(smiles_list):
        mol = Chem.MolFromSmiles(smi)
        if mol is None:
            continue
        scaf = MurckoScaffold.MurckoScaffoldSmiles(mol=mol, includeChirality=False)
        groups.setdefault(scaf, []).append(i)
    ordered = sorted(groups.values(), key=lambda g: (len(g), g[0]), reverse=True)
    n = sum(len(g) for g in ordered)
    train, valid, test = [], [], []
    for g in ordered:
        if len(train) + len(g) <= frac_train * n:
            train += g
        elif len(valid) + len(g) <= frac_valid * n:
            valid += g
        else:
            test += g
    return train, valid, test


def load_split_csv(name, csv_path):
    """Load a MoleculeNet CSV you already have and scaffold-split it locally."""
    smi_col, y_col = CSV_COLUMNS[name]
    df = pd.read_csv(csv_path)
    smiles, ys = df[smi_col].astype(str).tolist(), df[y_col].tolist()
    idx = scaffold_split(smiles)
    splits = {s: [(smiles[i], ys[i]) for i in ids]
              for s, ids in zip(("train", "valid", "test"), idx)}
    return splits, DATASETS[name][1]


CSV_PATHS = {}   # filled from --csv arguments


def load_split(name):
    """MoleculeNet via DeepChem with a scaffold split, or a local CSV if given."""
    if name in CSV_PATHS:
        return load_split_csv(name, CSV_PATHS[name])
    import deepchem as dc
    loader_name, task_type = DATASETS[name]
    tasks, (train, valid, test), _ = getattr(dc.molnet, loader_name)(
        featurizer="Raw", splitter="scaffold", transformers=[])
    splits = {}
    for split_name, ds in (("train", train), ("valid", valid), ("test", test)):
        y = ds.y[:, 0] if ds.y.ndim > 1 else ds.y      # BBBP, BACE, ESOL are single-task
        splits[split_name] = list(zip(ds.ids, y))
    return splits, task_type


def to_pyg(g, y, task_type):
    import torch
    from torch_geometric.data import Data
    edge_index = (torch.tensor(g["edges"], dtype=torch.long).t().contiguous()
                  if g["edges"] else torch.zeros((2, 0), dtype=torch.long))
    edge_dim = len(BOND_TYPES) + 1 + int(g["ionic"])
    edge_attr = (torch.tensor(g["edge_attr"], dtype=torch.float)
                 if g["edge_attr"] else torch.zeros((0, edge_dim)))
    return Data(
        x=torch.tensor(g["x"], dtype=torch.float),
        edge_index=edge_index,
        edge_attr=edge_attr,
        y=torch.tensor([float(y)], dtype=torch.float),
        smiles=g["smiles"],
        fragment_text=g["fragment_text"],
        atom_groups=g["atom_groups"],
        counterion=torch.tensor(g["counterion"], dtype=torch.bool),
        ionic=g["ionic"],
        label_version=label_version(),
        num_nodes=g["num_fragments"],
    )


# ----------------------------------------------------------------------------- main

def mode_paths(ionic):
    """Output locations for each mode, so the two variants never overwrite each other."""
    return ("frag_graphs_ions", "fragment_stats_ions.csv") if ionic else ("frag_graphs", "fragment_stats.csv")


def run(names, build, out_dir=None, ionic=False):
    default_dir, stats_path = mode_paths(ionic)
    out_dir = out_dir or default_dir
    rows = []
    for name in names:
        splits, task_type = load_split(name)
        counts, stripped, invalid, multi, ci_nodes, metal = [], 0, 0, 0, 0, 0
        for split_name, items in splits.items():
            graphs = []
            for smiles, y in items:
                g = fragment_graph(smiles, ionic)
                if g is None:
                    invalid += 1
                    continue
                counts.append(g["num_fragments"])
                stripped += g["salt_stripped"]
                multi += g["n_components"] > 1
                ci_nodes += g["n_counterion_nodes"]
                metal += g["metal_disconnected"]
                if build:
                    graphs.append(to_pyg(g, y, task_type))
            if build:
                import torch
                os.makedirs(out_dir, exist_ok=True)
                torch.save(graphs, os.path.join(out_dir, f"{name}_{split_name}.pt"))
                with open(os.path.join(out_dir, f"{name}_LABELS.txt"), "w") as f:
                    f.write(label_version())
        c = np.array(counts)
        row = {
            "dataset": name,
            "mode": "BRICS + ionic" if ionic else "BRICS (salts stripped)",
            "task": task_type,
            "molecules": len(c),
            "train/valid/test": "/".join(str(len(v)) for v in splits.values()),
            "mean_fragments": c.mean(),
            "std_fragments": c.std(),
            "median_fragments": float(np.median(c)),
            "max_fragments": int(c.max()),
            "pct_le_12": 100 * (c <= MAX_FRAGMENTS_FOR_EXACT_SHAPLEY).mean(),
            "pct_single_fragment": 100 * (c == 1).mean(),
            "salts_stripped": stripped,
            "invalid_smiles": invalid,
        }
        if ionic:
            row.update(pct_with_counterions=100 * multi / len(c), counterion_nodes=ci_nodes,
                       metal_bonds_split=metal)
        rows.append(row)
        print(f"done: {name}")

    df = pd.DataFrame(rows)
    if os.path.exists(stats_path):    # keep rows for datasets not rebuilt this time
        old = pd.read_csv(stats_path)
        df = pd.concat([old[~old["dataset"].isin(df["dataset"])], df], ignore_index=True)
    df.to_csv(stats_path, index=False)
    pd.set_option("display.width", 200)
    print("\n", df.round(2).to_string(index=False))
    print("\nLaTeX rows (Dataset & Task & #Mol & Frags mean±std & Max & % ≤12 & % single):")
    for r in rows:
        print(f"{r['dataset']} & {r['task']} & {r['molecules']} & "
              f"{r['mean_fragments']:.1f} $\\pm$ {r['std_fragments']:.1f} & {r['max_fragments']} & "
              f"{r['pct_le_12']:.1f} & {r['pct_single_fragment']:.1f} \\\\")


def demo(smiles, ionic=False):
    g = fragment_graph(smiles, ionic)
    if g is None:
        print("Invalid SMILES")
        return
    note = ("  (counter-ion removed)" if g["salt_stripped"] else
            f"  ({g['n_components']} components, {g['n_ionic_edges']} ionic edges)" if ionic else "")
    print(f"{g['smiles']}  ->  {g['num_fragments']} fragments{note}")
    for i, t in enumerate(g["fragment_text"]):
        print(f"  node {i}: {t}")
    print("  edges:", sorted({tuple(sorted(e)) + (("ionic",) if a[-1] == 1.0 and ionic else ())
                              for e, a in zip(g["edges"], g["edge_attr"])}))
    print("  node feature size:", g["x"].shape[1], "| edge feature size:", len(g["edge_attr"][0]) if g["edge_attr"] else "-")


if __name__ == "__main__":
    ap = argparse.ArgumentParser()
    ap.add_argument("--demo", type=str, help="print the fragment graph for one SMILES")
    ap.add_argument("--stats", action="store_true", help="fragment statistics for the datasets")
    ap.add_argument("--build", action="store_true", help="also save PyG graphs (needs torch_geometric)")
    ap.add_argument("--ionic", action="store_true",
                    help="BRICS + ionic decomposition (keep counter-ions as nodes); default strips them")
    ap.add_argument("--datasets", nargs="*", default=list(DATASETS))
    ap.add_argument("--out", default=None, help="graph folder (default frag_graphs or frag_graphs_ions)")
    ap.add_argument("--csv", nargs="*", default=[], metavar="NAME=PATH",
                    help="use local CSVs instead of DeepChem, e.g. BBBP=BBBP.csv BACE=bace.csv ESOL=delaney-processed.csv")
    args = ap.parse_args()
    for item in args.csv:
        k, v = item.split("=", 1)
        CSV_PATHS[k.upper()] = v
    if args.demo:
        demo(args.demo, args.ionic)
    elif args.stats or args.build:
        run(args.datasets, args.build, args.out, args.ionic)
    else:
        ap.print_help()

In [ ]:
%%writefile frag_gin.py
"""Fragment-graph GIN for molecular property prediction.

Fragment features -> MLP compression -> GINE layers (edge features = broken-bond type)
-> attention readout -> prediction head. `embed()` returns the fragment (node) and
molecule (graph) embeddings that the GSPELL projector consumes.
"""

import os
import random

import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch_geometric.loader import DataLoader
from torch_geometric.nn import GINEConv
from torch_geometric.nn.aggr import AttentionalAggregation
from sklearn.metrics import roc_auc_score


class FragGIN(nn.Module):
    def __init__(self, in_dim, edge_dim=5, hidden=128, layers=3, dropout=0.2):
        super().__init__()
        self.compress = nn.Sequential(nn.Linear(in_dim, hidden), nn.BatchNorm1d(hidden), nn.ReLU(),
                                      nn.Linear(hidden, hidden))
        self.convs, self.norms = nn.ModuleList(), nn.ModuleList()
        for _ in range(layers):
            mlp = nn.Sequential(nn.Linear(hidden, hidden), nn.BatchNorm1d(hidden), nn.ReLU(),
                                nn.Linear(hidden, hidden))
            self.convs.append(GINEConv(mlp, edge_dim=edge_dim))
            self.norms.append(nn.BatchNorm1d(hidden))
        self.readout = AttentionalAggregation(gate_nn=nn.Linear(hidden, 1))
        self.head = nn.Sequential(nn.Linear(hidden, hidden), nn.ReLU(), nn.Dropout(dropout), nn.Linear(hidden, 1))
        self.dropout = dropout

    def embed(self, data):
        """Fragment (node) embeddings and molecule (graph) embedding."""
        h = self.compress(data.x)
        for conv, norm in zip(self.convs, self.norms):
            h = h + F.dropout(F.relu(norm(conv(h, data.edge_index, data.edge_attr))), self.dropout, self.training)
        return h, self.readout(h, data.batch)

    def forward(self, data):
        return self.head(self.embed(data)[1]).squeeze(-1)


def load_graphs(name, graph_dir="frag_graphs"):
    return [torch.load(os.path.join(graph_dir, f"{name}_{s}.pt"), weights_only=False)
            for s in ("train", "valid", "test")]


def load_checkpoint(path, device):
    ckpt = torch.load(path, weights_only=False, map_location=device)
    model = FragGIN(in_dim=ckpt["in_dim"], edge_dim=ckpt.get("edge_dim", 5)).to(device)
    model.load_state_dict(ckpt["state_dict"])
    model.eval()
    return model, ckpt


@torch.no_grad()
def evaluate(model, loader, task, mu, sd, device):
    model.eval()
    preds, ys = [], []
    for b in loader:
        b = b.to(device)
        out = model(b)
        preds.append(torch.sigmoid(out) if task == "classification" else out * sd + mu)
        ys.append(b.y)
    p, y = torch.cat(preds).cpu().numpy(), torch.cat(ys).cpu().numpy()
    return roc_auc_score(y, p) if task == "classification" else float(np.sqrt(((p - y) ** 2).mean()))


def train_one(name, task, seed, device, graph_dir="frag_graphs", ckpt_dir="checkpoints",
              epochs=150, patience=25, lr=1e-3):
    """Train with early stopping on the validation metric; returns (best valid, test, epochs run)."""
    random.seed(seed); np.random.seed(seed); torch.manual_seed(seed); torch.cuda.manual_seed_all(seed)
    train, valid, test = load_graphs(name, graph_dir)
    ys = torch.cat([d.y for d in train])
    mu, sd = (ys.mean().item(), ys.std().item()) if task == "regression" else (0.0, 1.0)
    tl = DataLoader(train, batch_size=64, shuffle=True, drop_last=True)
    vl, te = DataLoader(valid, batch_size=256), DataLoader(test, batch_size=256)

    in_dim, edge_dim = train[0].x.size(1), train[0].edge_attr.size(1)   # 523/5 default, 525/6 ionic mode
    model = FragGIN(in_dim=in_dim, edge_dim=edge_dim).to(device)
    opt = torch.optim.Adam(model.parameters(), lr=lr, weight_decay=1e-5)
    better = (lambda a, b: a > b) if task == "classification" else (lambda a, b: a < b)
    best, best_state, wait, epoch = None, None, 0, 0
    for epoch in range(epochs):
        model.train()
        for b in tl:
            b = b.to(device)
            opt.zero_grad()
            out = model(b)
            loss = (F.binary_cross_entropy_with_logits(out, b.y) if task == "classification"
                    else F.mse_loss(out, (b.y - mu) / sd))
            loss.backward()
            opt.step()
        v = evaluate(model, vl, task, mu, sd, device)
        if best is None or better(v, best):
            best, wait = v, 0
            best_state = {k: t.detach().cpu().clone() for k, t in model.state_dict().items()}
        else:
            wait += 1
            if wait >= patience:
                break
    model.load_state_dict(best_state)
    test_metric = evaluate(model, te, task, mu, sd, device)
    os.makedirs(ckpt_dir, exist_ok=True)
    torch.save({"state_dict": best_state, "mu": mu, "sd": sd, "in_dim": in_dim, "edge_dim": edge_dim},
               os.path.join(ckpt_dir, f"{name}_gin_seed{seed}.pt"))
    return best, test_metric, epoch + 1

In [ ]:
%%writefile prepare_data.py
"""One command: get the MoleculeNet CSVs, scaffold-split them, build fragment graphs,
and write the statistics table.

    python prepare_data.py                     # downloads BBBP, BACE, ESOL into ./data
    python prepare_data.py --data_dir my_csvs  # use CSVs you already have

    python prepare_data.py --ionic             # keep counter-ions as nodes (BRICS + ionic)

Outputs: frag_graphs/{DATASET}_{train,valid,test}.pt and fragment_stats.csv
         (frag_graphs_ions/ and fragment_stats_ions.csv with --ionic)
"""

import argparse
import os
import urllib.request

import pandas as pd

import fragment_graphs as fg

URLS = {
    "BBBP": "https://deepchemdata.s3-us-west-1.amazonaws.com/datasets/BBBP.csv",
    "BACE": "https://deepchemdata.s3-us-west-1.amazonaws.com/datasets/bace.csv",
    "ESOL": "https://deepchemdata.s3-us-west-1.amazonaws.com/datasets/delaney-processed.csv",
}
# File names tried, in order, when --data_dir is given
LOCAL_NAMES = {
    "BBBP": ["BBBP.csv", "bbbp.csv"],
    "BACE": ["bace.csv", "BACE.csv"],
    "ESOL": ["esol.csv", "ESOL.csv", "delaney-processed.csv", "delaney.csv"],
}
LABEL_CANDIDATES = {
    "BBBP": ["p_np", "label", "y"],
    "BACE": ["Class", "class", "label", "y"],
    "ESOL": ["measured log solubility in mols per litre", "logSolubility", "measured_log_solubility",
             "logS", "solubility", "y"],
}


def locate(name, data_dir):
    for fname in LOCAL_NAMES[name]:
        path = os.path.join(data_dir, fname)
        if os.path.exists(path):
            return path
    os.makedirs(data_dir, exist_ok=True)
    path = os.path.join(data_dir, os.path.basename(URLS[name]))
    print(f"downloading {name} -> {path}")
    urllib.request.urlretrieve(URLS[name], path)
    return path


def detect_columns(name, path):
    cols = list(pd.read_csv(path, nrows=1).columns)
    smi = next((c for c in cols if c.lower() in ("smiles", "mol")), None)
    lab = next((c for c in LABEL_CANDIDATES[name] if c in cols), None)
    if smi is None or lab is None:
        raise KeyError(f"{name}: can't find SMILES/label columns in {path}; header is {cols}")
    return smi, lab


if __name__ == "__main__":
    ap = argparse.ArgumentParser()
    ap.add_argument("--data_dir", default="data")
    ap.add_argument("--datasets", nargs="*", default=list(fg.DATASETS))
    ap.add_argument("--ionic", action="store_true", help="BRICS + ionic decomposition -> frag_graphs_ions/")
    ap.add_argument("--out", default=None)
    args = ap.parse_args()
    for name in args.datasets:
        path = locate(name, args.data_dir)
        fg.CSV_COLUMNS[name] = detect_columns(name, path)
        fg.CSV_PATHS[name] = path
        print(f"{name}: {path}  columns={fg.CSV_COLUMNS[name]}")
    fg.run(args.datasets, build=True, out_dir=args.out, ionic=args.ionic)

In [ ]:
%%writefile gspell_mol.py
"""GSPELL adapted to graph-level explanation on fragment graphs.

Original method (Baghershahi et al., arXiv:2508.07117) explains NODE predictions on
text-attributed graphs. Here the target is the MOLECULE: its graph embedding is the
target soft prompt, and its BRICS fragments play the role of the neighbors whose
support the LLM judges.

Steps
  1. Frozen fragment-GIN gives fragment embeddings h_v and molecule embeddings h_G.
  2. A projector maps each embedding to k soft-prompt tokens in the LLM's embedding
     space, trained with GSPELL's context-alignment + contrastive losses.
  3. A hybrid prompt interleaves soft prompts with fragment text; the frozen LLM
     returns YES/NO support per fragment. YES fragments form the explanation subgraph.
  4. Fidelity (does the GIN give the same prediction on the explanation subgraph?)
     and size are compared with GNNExplainer, a random baseline, and a text-only LLM.
"""

import json
import math
import random
import re

import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch_geometric.data import Data
from torch_geometric.loader import DataLoader
from torch_geometric.utils import subgraph

TASK_TEXT = {
    "BBBP": ("penetrates the blood-brain barrier", "does not penetrate the blood-brain barrier"),
    "BACE": ("inhibits BACE-1", "does not inhibit BACE-1"),
    # Regression: the prediction is turned into a class at the training-set median
    "ESOL": ("is more water-soluble than the median compound in the dataset",
             "is less water-soluble than the median compound in the dataset"),
}

SYSTEM_PROMPT = (
    "You are a medicinal chemist explaining the prediction of a graph neural network (GNN). "
    "The GNN represents a molecule as a graph of chemical fragments. Each fragment is shown with "
    "its SMILES, its functional groups, and the GNN's internal embedding of it."
)


# ----------------------------------------------------------------------------- LLM

def load_llm(model_id="meta-llama/Llama-3.1-8B-Instruct", low_memory=None):
    """bf16 on GPUs with >= 22 GB and bf16 support (L4, A100); 4-bit fp16 otherwise (T4)."""
    from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig
    if low_memory is None:
        cap = torch.cuda.get_device_capability()
        mem = torch.cuda.get_device_properties(0).total_memory / 1e9
        low_memory = cap[0] < 8 or mem < 22
    tok = AutoTokenizer.from_pretrained(model_id)
    if tok.pad_token is None:
        tok.pad_token = tok.eos_token
    if low_memory:
        llm = AutoModelForCausalLM.from_pretrained(
            model_id, device_map="auto", torch_dtype=torch.float16,
            quantization_config=BitsAndBytesConfig(load_in_4bit=True, bnb_4bit_quant_type="nf4",
                                                   bnb_4bit_compute_dtype=torch.float16))
    else:
        llm = AutoModelForCausalLM.from_pretrained(model_id, device_map="auto", torch_dtype=torch.bfloat16)
    llm.eval()
    return tok, llm


@torch.no_grad()
def text_embeddings(texts, tok, llm, batch_size=256):
    """LLM(v) in GSPELL's context loss: mean of the LLM's input token embeddings of the text."""
    emb = llm.get_input_embeddings()
    out = []
    for i in range(0, len(texts), batch_size):
        enc = tok(texts[i:i + batch_size], return_tensors="pt", padding=True, add_special_tokens=False)
        ids = enc.input_ids.to(emb.weight.device)
        mask = enc.attention_mask.to(emb.weight.device).unsqueeze(-1).float()
        e = emb(ids).float()
        out.append(((e * mask).sum(1) / mask.sum(1).clamp(min=1)).cpu())
    return torch.cat(out)


# ----------------------------------------------------------------------------- projector

class Projector(nn.Module):
    """GNN embedding -> k soft tokens. Outputs are rescaled to the LLM's average
    token-embedding norm so the soft prompts sit in the same range as real tokens."""

    def __init__(self, in_dim, llm_dim, k=5, hidden=512, token_norm=1.0):
        super().__init__()
        self.k, self.llm_dim = k, llm_dim
        self.net = nn.Sequential(nn.Linear(in_dim, hidden), nn.GELU(), nn.Linear(hidden, k * llm_dim))
        self.register_buffer("token_norm", torch.tensor(float(token_norm)))

    def forward(self, h):
        z = self.net(h).view(-1, self.k, self.llm_dim)
        return F.normalize(z, dim=-1) * self.token_norm


@torch.no_grad()
def collect_embeddings(gin, graphs, device, batch_size=128):
    """Node embeddings + fragment text, and graph embeddings + molecule text."""
    gin.eval()
    H_nodes, T_nodes, H_graphs, T_graphs = [], [], [], []
    for b in DataLoader(graphs, batch_size=batch_size):
        h, g = gin.embed(b.to(device))
        H_nodes.append(h.cpu()); H_graphs.append(g.cpu())
        T_nodes += [t for texts in b.fragment_text for t in texts]
        T_graphs += [f"molecule {s}" for s in b.smiles]
    return torch.cat(H_nodes), T_nodes, torch.cat(H_graphs), T_graphs


def train_projector(H, T, llm_dim, token_norm, device, k=5, epochs=30, batch_size=512,
                    tau=0.2, w_context=1.0, w_contrast=0.5, lr=1e-3, seed=0):
    """GSPELL losses: L = w_context * L_context + w_contrast * L_contrast."""
    torch.manual_seed(seed)
    proj = Projector(H.size(1), llm_dim, k=k, token_norm=token_norm).to(device)
    opt = torch.optim.Adam(proj.parameters(), lr=lr)
    H, T = H.to(device), F.normalize(T.to(device), dim=-1)
    history = []
    for epoch in range(epochs):
        perm = torch.randperm(H.size(0), device=device)
        total = 0.0
        for i in range(0, H.size(0), batch_size):
            idx = perm[i:i + batch_size]
            zbar = F.normalize(proj(H[idx]).mean(1), dim=-1)
            l_ctx = -(zbar * T[idx]).sum(-1).mean()
            hn = F.normalize(H[idx], dim=-1)
            p_gnn = F.softmax(hn @ hn.T / tau, dim=-1)
            log_p_proj = F.log_softmax(zbar @ zbar.T / tau, dim=-1)
            l_con = -(p_gnn * log_p_proj).sum(-1).mean()
            loss = w_context * l_ctx + w_contrast * l_con
            opt.zero_grad(); loss.backward(); opt.step()
            total += loss.item() * len(idx)
        history.append(total / H.size(0))
    proj.eval()
    return proj, history


# ----------------------------------------------------------------------------- prompting

def build_user_prompt(data, label_text, with_soft=True):
    z = lambda key: f" <<Z_{key}>>" if with_soft else ""
    lines = [f"Target molecule:{z('G')}",
             f"GNN prediction: the molecule {label_text}.",
             "Fragments of the molecule:"]
    for i, t in enumerate(data.fragment_text):
        lines.append(f"Fragment {i + 1}: {t}{z(i)}")
    lines += ["",
              "For each fragment, write exactly one line in this format:",
              "Fragment <number>: Summary: <one sentence>. Support: YES or NO",
              f"YES means the fragment supports the prediction that the molecule {label_text}.",
              "Only use the fragment numbers listed above. After the fragments, write one line "
              "starting with 'Reasoning:' that explains the prediction in one or two sentences."]
    return "\n".join(lines)


@torch.no_grad()
def generate(tok, llm, user_prompt, soft=None, max_new_tokens=400):
    """Chat-formatted prompt with <<Z_key>> markers replaced by soft-prompt tensors."""
    emb = llm.get_input_embeddings()
    dev, dtype = emb.weight.device, emb.weight.dtype
    text = tok.apply_chat_template([{"role": "system", "content": SYSTEM_PROMPT},
                                    {"role": "user", "content": user_prompt}],
                                   tokenize=False, add_generation_prompt=True)
    pieces = []
    for part in re.split(r"(<<Z_\w+>>)", text):
        m = re.fullmatch(r"<<Z_(\w+)>>", part)
        if m:
            pieces.append(soft[m.group(1)].to(dev, dtype))
        elif part:
            ids = tok(part, add_special_tokens=False, return_tensors="pt").input_ids.to(dev)
            pieces.append(emb(ids)[0])
    inputs = torch.cat(pieces).unsqueeze(0)
    out = llm.generate(inputs_embeds=inputs, attention_mask=torch.ones(inputs.shape[:2], device=dev, dtype=torch.long),
                       max_new_tokens=max_new_tokens, do_sample=False, pad_token_id=tok.eos_token_id)
    return tok.decode(out[0], skip_special_tokens=True)


def parse_support(text, n_frag):
    """Map fragment index -> True (YES) / False (NO). Out-of-range numbers are dropped
    (GSPELL's post-hoc hallucination filter) and counted."""
    support, hallucinated = {}, 0
    for line in text.splitlines():
        m = re.match(r"\W*Fragment\s*(\d+)\b(.*)", line, flags=re.I)
        if not m:
            continue
        s = re.search(r"Support\W*\s*(YES|NO)\b", m.group(2), flags=re.I)
        if not s:
            continue
        idx = int(m.group(1)) - 1
        if 0 <= idx < n_frag:
            support[idx] = s.group(1).upper() == "YES"
        else:
            hallucinated += 1
    return support, hallucinated


# ----------------------------------------------------------------------------- evaluation

class ModelSpec:
    """A trained fragment-GIN plus what's needed to turn its output into a class:
    probability >= 0.5 for classification, value >= training median for regression."""

    def __init__(self, gin, task, mu=0.0, sd=1.0, threshold=0.5):
        self.gin, self.task, self.mu, self.sd, self.threshold = gin, task, mu, sd, threshold

    def score(self, out):
        return torch.sigmoid(out) if self.task == "classification" else out * self.sd + self.mu


def load_spec(dataset, seed, device, graph_dir="frag_graphs", ckpt_dir="checkpoints"):
    from fragment_graphs import DATASETS
    from frag_gin import load_checkpoint, load_graphs
    gin, ckpt = load_checkpoint(f"{ckpt_dir}/{dataset}_gin_seed{seed}.pt", device)
    task = DATASETS[dataset][1]
    threshold = 0.5
    if task == "regression":
        train = load_graphs(dataset, graph_dir)[0]
        threshold = float(torch.cat([d.y for d in train]).median())
    return ModelSpec(gin, task, ckpt["mu"], ckpt["sd"], threshold)


def canonical_graph(data):
    """Rebuild a molecule's fragment graph from its stored (canonical) SMILES so that
    `atom_groups` index the atoms of Chem.MolFromSmiles(data.smiles). Needed to map
    fragment scores onto atoms and to align with atom-level models."""
    from fragment_graphs import fragment_graph, to_pyg
    g = fragment_graph(data.smiles, ionic=bool(getattr(data, "ionic", False)))
    g["smiles"] = data.smiles     # atom_groups index the atoms of exactly this string
    return to_pyg(g, float(data.y), None)


@torch.no_grad()
def predict(spec, data, device, keep=None):
    """Model score (probability or value) on the full graph or on the subgraph of `keep` fragments."""
    if keep is None:
        d = Data(x=data.x, edge_index=data.edge_index, edge_attr=data.edge_attr)
    else:
        keep_t = torch.tensor(sorted(keep), dtype=torch.long)
        ei, ea = subgraph(keep_t, data.edge_index, data.edge_attr, relabel_nodes=True, num_nodes=data.num_nodes)
        d = Data(x=data.x[keep_t], edge_index=ei, edge_attr=ea)
    d.batch = torch.zeros(d.x.size(0), dtype=torch.long)
    spec.gin.eval()   # BatchNorm must use running stats: explanation subgraphs can have one node
    return spec.score(spec.gin(d.to(device))).item()


def fidelity(spec, data, keep, full_score, device):
    if not keep:
        return 0.0        # an empty explanation cannot reproduce a prediction
    return float((predict(spec, data, device, keep) >= spec.threshold) == (full_score >= spec.threshold))


class _GraphWrap(nn.Module):
    def __init__(self, gin):
        super().__init__()
        self.gin = gin

    def forward(self, x, edge_index, edge_attr, batch):
        d = Data(x=x, edge_index=edge_index, edge_attr=edge_attr)
        d.batch = batch
        return self.gin(d).view(-1, 1)


def gnnexplainer_mask(spec, data, device, epochs=200):
    from torch_geometric.explain import Explainer, GNNExplainer
    mode = "binary_classification" if spec.task == "classification" else "regression"
    explainer = Explainer(
        model=_GraphWrap(spec.gin).eval(), algorithm=GNNExplainer(epochs=epochs),
        explanation_type="model", node_mask_type="object", edge_mask_type=None,
        model_config=dict(mode=mode, task_level="graph", return_type="raw"))
    d = data.clone().to(device)
    batch = torch.zeros(d.num_nodes, dtype=torch.long, device=device)
    exp = explainer(d.x, d.edge_index, edge_attr=d.edge_attr, batch=batch)
    spec.gin.eval()   # the Explainer restores the wrapper's training flag afterwards
    return exp.node_mask.view(-1).detach().cpu()


def top_k(scores, k):
    return torch.as_tensor(scores).argsort(descending=True)[:k].tolist()


def _done_smiles(path):
    import os
    if not path or not os.path.exists(path):
        return []
    with open(path) as f:
        return [json.loads(line) for line in f if line.strip()]


def run_explanations(spec, proj, tok, llm, graphs, dataset, device, n_eval=30, min_frag=2,
                     max_frag=12, text_only_baseline=True, extra_explainers=None, seed=0,
                     log_every=5, records_path=None, only_counterions=False, indices=None):
    """Explain n_eval test molecules.

    extra_explainers: {name: fn(spec, data, device) -> per-fragment scores}. Each is
    evaluated at GSPELL's explanation size (top-k). Scores are stored for agreement analysis.
    only_counterions: ionic mode; explain only molecules that have counter-ion nodes.
    indices: explain exactly graphs[i] for these i (used to explain the SAME molecules in
    both fragmentation modes); overrides n_eval, min/max_frag and only_counterions.
    records_path: a .jsonl file; finished molecules are appended as they complete, and
    molecules already in the file are skipped, so an interrupted run can be resumed.
    Returns (per-molecule records, summary table)."""
    pos_text, neg_text = TASK_TEXT[dataset]
    extra_explainers = extra_explainers or {}
    rng = random.Random(seed)
    if indices is None:
        pool = [i for i, d in enumerate(graphs) if min_frag <= d.num_nodes <= max_frag]
        if only_counterions:
            pool = [i for i in pool if hasattr(graphs[i], "counterion") and bool(graphs[i].counterion.any())]
        random.Random(seed).shuffle(pool)  # fixed order: raising n_eval later extends the same list
        indices = pool[:n_eval]
    chosen = [graphs[i] for i in indices]
    wanted = {d.smiles for d in chosen}
    index_of = {d.smiles: int(i) for i, d in zip(indices, chosen)}
    records = _done_smiles(records_path)
    for r in records:                 # entries written by older versions lack these fields
        r.setdefault("test_index", index_of.get(r["smiles"]))
        r.setdefault("counterion", [False] * r["n_fragments"])
    done = {r["smiles"] for r in records}
    if done:
        print(f"  resuming: {len(done)} molecules already explained")
    for j, (test_index, data) in enumerate(zip(indices, chosen)):
        if data.smiles in done:
            continue
        data = canonical_graph(data.cpu())
        n = data.num_nodes
        full = predict(spec, data, device)
        label_text = pos_text if full >= spec.threshold else neg_text
        rec = {"smiles": data.smiles, "test_index": int(test_index), "n_fragments": n, "gin_score": full, "y": float(data.y),
               "fragments": list(data.fragment_text), "atom_groups": data.atom_groups,
               "counterion": data.counterion.tolist() if hasattr(data, "counterion") else [False] * n}

        # GSPELL (soft prompts)
        with torch.no_grad():
            d = data.clone().to(device)
            d.batch = torch.zeros(n, dtype=torch.long, device=device)
            h, g = spec.gin.embed(d)
            zs = proj(torch.cat([h, g]))
        soft = {str(i): zs[i] for i in range(n)}
        soft["G"] = zs[n]
        out = generate(tok, llm, build_user_prompt(data, label_text, True), soft)
        sup, hal = parse_support(out, n)
        keep = sorted(i for i, v in sup.items() if v)
        k = max(1, len(keep))
        rec.update(gspell_output=out, gspell_keep=keep, gspell_hallucinated=hal,
                   gspell_fid=fidelity(spec, data, keep, full, device), gspell_size=len(keep))

        # Text-only LLM (no projector)
        if text_only_baseline:
            out_t = generate(tok, llm, build_user_prompt(data, label_text, False))
            sup_t, _ = parse_support(out_t, n)
            keep_t = sorted(i for i, v in sup_t.items() if v)
            rec.update(llm_output=out_t, llm_keep=keep_t,
                       llm_fid=fidelity(spec, data, keep_t, full, device), llm_size=len(keep_t))

        # GNNExplainer: its own threshold, and size-matched to GSPELL
        mask = gnnexplainer_mask(spec, data, device)
        keep_g = [i for i in range(n) if mask[i] > 0.5] or [int(mask.argmax())]
        keep_gk = top_k(mask, k)
        rec.update(gnnexp_scores=mask.tolist(), gnnexp_keep=keep_g,
                   gnnexp_fid=fidelity(spec, data, keep_g, full, device), gnnexp_size=len(keep_g),
                   gnnexp_k_keep=keep_gk, gnnexp_k_fid=fidelity(spec, data, keep_gk, full, device),
                   gnnexp_k_size=k)

        # Gradient / attention explainers, size-matched to GSPELL
        for name, fn in extra_explainers.items():
            scores = torch.as_tensor(fn(spec, data, device), dtype=torch.float).view(-1)
            keep_e = top_k(scores, k)
            rec.update({f"{name}_scores": scores.tolist(), f"{name}_keep": keep_e,
                        f"{name}_fid": fidelity(spec, data, keep_e, full, device), f"{name}_size": k})

        # Random: half the fragments, 5 draws
        r_f = [fidelity(spec, data, rng.sample(range(n), math.ceil(n / 2)), full, device) for _ in range(5)]
        rec.update(random_fid=float(np.mean(r_f)), random_size=math.ceil(n / 2))

        records.append(rec)
        if records_path:
            with open(records_path, "a") as f:
                f.write(json.dumps(rec, default=str) + "\n")
        if (j + 1) % log_every == 0:
            print(f"  explained {j + 1}/{len(chosen)}")

    df = pd.DataFrame([r for r in records if r["smiles"] in wanted])   # ignore stale entries in the file
    return df, summarize(df, extra_explainers)


METHOD_NAMES = {"gspell": "GSPELL (graph-level)", "llm": "LLM text-only (no projector)",
                "gnnexp": "GNNExplainer", "gnnexp_k": "GNNExplainer (size-matched)",
                "frag_gradcam": "Grad-CAM, fragment GIN (size-matched)",
                "attention": "Readout attention (size-matched)",
                "atom_gradcam": "Grad-CAM, atom GIN -> fragments (size-matched)",
                "random": "Random (half)"}


def paired_indices(graph_lists, n_eval, seed=0, min_frag=2, max_frag=12, require_counterion=None):
    """Test-set positions that are explainable in every mode, in a fixed shuffled order.
    graph_lists: {mode: list of test graphs}; all modes come from the same CSV and scaffold
    split, so position i is the same molecule in each. require_counterion: a mode name whose
    graphs must contain a counter-ion node (to focus on salts)."""
    lists = list(graph_lists.values())
    n = len(lists[0])
    if any(len(g) != n for g in lists):
        raise ValueError("modes have different test-set sizes; rebuild both from the same CSVs")
    ok = [i for i in range(n) if all(min_frag <= g[i].num_nodes <= max_frag for g in lists)]
    if require_counterion:
        ok = [i for i in ok if bool(graph_lists[require_counterion][i].counterion.any())]
    random.Random(seed).shuffle(ok)
    return ok[:n_eval]


def summarize(df, extra_explainers=None):
    keys = ["gspell", "llm", "gnnexp", "gnnexp_k"] + list(extra_explainers or {}) + ["random"]
    return pd.DataFrame([
        {"method": METHOD_NAMES.get(key, key), "fidelity_%": 100 * df[f"{key}_fid"].mean(),
         "fidelity_std": 100 * df[f"{key}_fid"].std(), "size": df[f"{key}_size"].mean(),
         "size_frac_%": 100 * (df[f"{key}_size"] / df["n_fragments"]).mean(), "n": len(df)}
        for key in keys if f"{key}_fid" in df])


def save_records(df, path):
    with open(path, "w") as f:
        json.dump(df.to_dict(orient="records"), f, indent=1, default=str)

In [ ]:
%%writefile cam_mol.py
"""Gradient- and attention-based explanations (CAM family) for fragment and atom graphs,
plus RDKit drawings.

Three per-fragment score functions, all with signature fn(spec, data, device) -> tensor[n]:
  frag_gradcam   Grad-CAM (Selvaraju et al. 2017; for graphs, Pope et al. 2019) on the
                 fragment GIN's last-layer fragment embeddings.
  attention      the fragment GIN's attention-readout weights (where the model pools from).
  atom_gradcam   Grad-CAM on an atom-level GIN trained on the same split, summed over each
                 fragment's atoms. Lets atom- and fragment-level models be compared on the
                 same units.

Scores are signed toward the PREDICTED class: for a negative prediction (or a regression
value below the threshold) the gradient of the negated output is used, so a high score
always means "supports this prediction".

The classic CAM of Zhou et al. (2016) needs global average pooling followed by a linear
layer; the fragment GIN uses attention pooling and an MLP head, so Grad-CAM is used instead.
"""

import os
import random

import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch_geometric.data import Data
from torch_geometric.loader import DataLoader
from torch_geometric.nn import GINEConv, global_add_pool
from torch_geometric.nn.aggr import AttentionalAggregation
from torch_geometric.utils import softmax as segment_softmax


# ----------------------------------------------------------------------------- fragment GIN

def _one_graph(data, device):
    d = Data(x=data.x, edge_index=data.edge_index, edge_attr=data.edge_attr).to(device)
    d.batch = torch.zeros(d.x.size(0), dtype=torch.long, device=device)
    return d


def _signed(spec, raw_out):
    """Output oriented toward the predicted class."""
    pred_pos = spec.score(raw_out.detach()).item() >= spec.threshold
    return raw_out.sum() if pred_pos else -raw_out.sum()


def gradcam(h, target):
    """Grad-CAM over nodes: channel weights = mean gradient over nodes; score = ReLU(sum_c a_c h_vc).
    Returns the pre-ReLU map (used for ranking, avoids ties at zero)."""
    grads = torch.autograd.grad(target, h)[0]
    alpha = grads.mean(dim=0)
    return (h * alpha).sum(dim=1).detach().cpu()


def frag_gradcam(spec, data, device):
    gin = spec.gin.eval()
    with torch.enable_grad():
        h, g = gin.embed(_one_graph(data, device))
        return gradcam(h, _signed(spec, gin.head(g)))


@torch.no_grad()
def attention(spec, data, device):
    gin = spec.gin.eval()
    d = _one_graph(data, device)
    h, _ = gin.embed(d)
    gate = gin.readout.gate_nn(h).view(-1)
    return segment_softmax(gate, d.batch).cpu()


# ----------------------------------------------------------------------------- atom GIN

def atom_graph(smiles, y=0.0):
    """PyG's standard RDKit atom graph (9 categorical atom features, 3 bond features).
    Atom order matches Chem.MolFromSmiles(smiles). Returns None if featurization fails."""
    from torch_geometric.utils import from_smiles
    try:
        d = from_smiles(smiles)
    except Exception:
        return None
    if d.x is None or d.x.size(0) == 0:
        return None
    d.y = torch.tensor([float(y)], dtype=torch.float)
    return d


def _vocab_sizes():
    from torch_geometric.utils.smiles import e_map, x_map
    return [len(v) for v in x_map.values()], [len(v) for v in e_map.values()]


class CatEncoder(nn.Module):
    """Sum of one embedding per categorical column (like OGB's AtomEncoder)."""

    def __init__(self, sizes, dim):
        super().__init__()
        self.embs = nn.ModuleList(nn.Embedding(s + 1, dim) for s in sizes)

    def forward(self, x):
        x = x.long()
        return sum(emb(x[:, i].clamp(max=emb.num_embeddings - 1)) for i, emb in enumerate(self.embs))


class AtomGIN(nn.Module):
    def __init__(self, hidden=128, layers=4, dropout=0.2):
        super().__init__()
        xs, es = _vocab_sizes()
        self.atom_enc = CatEncoder(xs, hidden)
        self.convs, self.norms, self.bond_encs = nn.ModuleList(), nn.ModuleList(), nn.ModuleList()
        for _ in range(layers):
            mlp = nn.Sequential(nn.Linear(hidden, hidden), nn.BatchNorm1d(hidden), nn.ReLU(), nn.Linear(hidden, hidden))
            self.convs.append(GINEConv(mlp))
            self.norms.append(nn.BatchNorm1d(hidden))
            self.bond_encs.append(CatEncoder(es, hidden))
        self.readout = AttentionalAggregation(gate_nn=nn.Linear(hidden, 1))
        self.head = nn.Sequential(nn.Linear(hidden, hidden), nn.ReLU(), nn.Dropout(dropout), nn.Linear(hidden, 1))
        self.dropout = dropout

    def embed(self, data):
        h = self.atom_enc(data.x)
        for conv, norm, be in zip(self.convs, self.norms, self.bond_encs):
            e = be(data.edge_attr) if data.edge_attr is not None and data.edge_attr.numel() else None
            if e is None:
                e = torch.zeros(data.edge_index.size(1), h.size(1), device=h.device)
            h = h + F.dropout(F.relu(norm(conv(h, data.edge_index, e))), self.dropout, self.training)
        return h, self.readout(h, data.batch)

    def forward(self, data):
        return self.head(self.embed(data)[1]).squeeze(-1)


def build_atom_splits(frag_splits):
    """Atom graphs for the same molecules and the same scaffold split as the fragment graphs."""
    out = []
    for split in frag_splits:
        gs = [atom_graph(d.smiles, float(d.y)) for d in split]
        out.append([g for g in gs if g is not None])
    return out


def train_atom_gin(name, task, splits, seed, device, ckpt_dir="checkpoints", epochs=150, patience=25, lr=1e-3):
    """Same protocol as frag_gin.train_one; returns (valid metric, test metric, model, mu, sd)."""
    from frag_gin import evaluate
    random.seed(seed); np.random.seed(seed); torch.manual_seed(seed); torch.cuda.manual_seed_all(seed)
    train, valid, test = splits
    ys = torch.cat([d.y for d in train])
    mu, sd = (ys.mean().item(), ys.std().item()) if task == "regression" else (0.0, 1.0)
    tl = DataLoader(train, batch_size=64, shuffle=True, drop_last=True)
    vl, te = DataLoader(valid, batch_size=256), DataLoader(test, batch_size=256)
    model = AtomGIN().to(device)
    opt = torch.optim.Adam(model.parameters(), lr=lr, weight_decay=1e-5)
    better = (lambda a, b: a > b) if task == "classification" else (lambda a, b: a < b)
    best, best_state, wait = None, None, 0
    for _ in range(epochs):
        model.train()
        for b in tl:
            b = b.to(device)
            opt.zero_grad()
            out = model(b)
            loss = (F.binary_cross_entropy_with_logits(out, b.y) if task == "classification"
                    else F.mse_loss(out, (b.y - mu) / sd))
            loss.backward()
            opt.step()
        v = evaluate(model, vl, task, mu, sd, device)
        if best is None or better(v, best):
            best, wait = v, 0
            best_state = {k: t.detach().cpu().clone() for k, t in model.state_dict().items()}
        else:
            wait += 1
            if wait >= patience:
                break
    model.load_state_dict(best_state)
    model.eval()
    test_metric = evaluate(model, te, task, mu, sd, device)
    os.makedirs(ckpt_dir, exist_ok=True)
    torch.save({"state_dict": best_state, "mu": mu, "sd": sd, "valid": best, "test": test_metric},
               f"{ckpt_dir}/{name}_atomgin_seed{seed}.pt")
    return best, test_metric, model, mu, sd


def load_atom_gin(path, device):
    """Reload a trained atom GIN. Returns (model, mu, sd, valid metric, test metric);
    the metrics are None for checkpoints saved before they were recorded."""
    ckpt = torch.load(path, weights_only=False, map_location=device)
    model = AtomGIN().to(device)
    model.load_state_dict(ckpt["state_dict"])
    model.eval()
    return model, ckpt["mu"], ckpt["sd"], ckpt.get("valid"), ckpt.get("test")


def atom_gradcam_scores(atom_spec, smiles, device):
    """Per-atom Grad-CAM for the atom GIN (pre-ReLU), oriented toward its own predicted class."""
    g = atom_graph(smiles)
    d = g.to(device)
    d.batch = torch.zeros(d.num_nodes, dtype=torch.long, device=device)
    model = atom_spec.gin.eval()
    with torch.enable_grad():
        h, pooled = model.embed(d)
        return gradcam(h, _signed(atom_spec, model.head(pooled)))


def make_atom_gradcam(atom_spec):
    """Returns fn(spec, data, device) -> per-fragment scores: atom Grad-CAM summed over each
    fragment's atoms. `data` must come from canonical_graph() so atom indices line up."""
    def fn(spec, data, device):
        a = atom_gradcam_scores(atom_spec, data.smiles, device)
        return torch.stack([a[list(group)].sum() for group in data.atom_groups])
    return fn


# ----------------------------------------------------------------------------- drawing
#
# Two figure types:
#   explanation_grid   one row per molecule, one panel per method. Fragments are shown as
#                      whole units: atoms AND the bonds inside each fragment are colored,
#                      the bonds BRICS cut are left gray, and each fragment is numbered F1..Fn.
#   fragment_panels    one row per molecule: the molecule (fragments numbered and outlined by
#                      color) followed by each fragment drawn on its own, with its attachment
#                      points, colored by importance and captioned with every method's score.
#
# Colors: red supports the prediction, blue opposes it, scaled per molecule.

CUT_BOND_COLOR = (0.75, 0.75, 0.75)


def _colors(values, cmap="RdBu_r"):
    import matplotlib
    v = np.asarray(values, dtype=float)
    lim = max(np.abs(v).max(), 1e-8)
    cm = matplotlib.colormaps[cmap]
    return [tuple(cm(0.5 + 0.45 * x / lim)[:3]) for x in v]


def _frag_index(atom_groups):
    return {a: f for f, g in enumerate(atom_groups) for a in g}


def _cut_bonds(mol, atom_groups):
    """Bonds whose two atoms sit in different fragments (the BRICS cuts)."""
    fi = _frag_index(atom_groups)
    return [b.GetIdx() for b in mol.GetBonds() if fi[b.GetBeginAtomIdx()] != fi[b.GetEndAtomIdx()]]


def _png(drawer):
    drawer.FinishDrawing()
    return drawer.GetDrawingText()


def _drawer(size, legend_size=15):
    from rdkit.Chem.Draw import rdMolDraw2D
    d = rdMolDraw2D.MolDraw2DCairo(*size)
    o = d.drawOptions()
    o.legendFontSize = legend_size
    o.annotationFontScale = 0.8
    o.fillHighlights = True
    o.continuousHighlight = True
    o.highlightBondWidthMultiplier = 14
    return d


def draw_fragment_map(smiles, atom_groups, frag_scores=None, atom_scores=None, keep=None,
                      size=(340, 280), legend="", number=True):
    """Draw a molecule with fragment structure visible.

    frag_scores: one value per fragment -> the fragment's atoms and internal bonds share its color.
    atom_scores: one value per atom (atom-level Grad-CAM) -> atoms colored individually; each
                 internal bond gets the mean of its two atoms.
    keep:        list of fragment indices -> those fragments in red, others uncolored (GSPELL).
    Cut bonds stay gray in every case, so fragment boundaries are visible."""
    from rdkit import Chem
    from rdkit.Chem.Draw import rdMolDraw2D
    mol = Chem.Mol(Chem.MolFromSmiles(smiles))
    fi = _frag_index(atom_groups)
    if keep is not None:
        frag_scores = [1.0 if f in set(keep) else 0.0 for f in range(len(atom_groups))]
    if frag_scores is not None:
        fc = _colors(frag_scores)
        atom_col = {a: fc[fi[a]] for a in range(mol.GetNumAtoms())}
        if keep is not None:   # unselected fragments: no color at all
            atom_col = {a: c for a, c in atom_col.items() if fi[a] in set(keep)}
    else:
        ac = _colors(atom_scores)
        atom_col = {a: ac[a] for a in range(mol.GetNumAtoms())}

    bond_col = {}
    for b in mol.GetBonds():
        i, j = b.GetBeginAtomIdx(), b.GetEndAtomIdx()
        if fi[i] != fi[j]:
            bond_col[b.GetIdx()] = CUT_BOND_COLOR
        elif i in atom_col and j in atom_col:
            if atom_scores is not None and frag_scores is None:
                bond_col[b.GetIdx()] = tuple((np.array(atom_col[i]) + np.array(atom_col[j])) / 2)
            else:
                bond_col[b.GetIdx()] = atom_col[i]

    if number:   # label each fragment once, on its first atom
        for f, g in enumerate(atom_groups):
            mol.GetAtomWithIdx(int(g[0])).SetProp("atomNote", f"F{f + 1}")

    d = _drawer(size)
    rdMolDraw2D.PrepareAndDrawMolecule(
        d, mol, legend=legend,
        highlightAtoms=list(atom_col), highlightAtomColors=atom_col,
        highlightBonds=list(bond_col), highlightBondColors=bond_col,
        highlightAtomRadii={a: 0.4 for a in atom_col})
    return _png(d)


def fragment_mols(smiles, atom_groups):
    """Each fragment as its own molecule with dummy atoms (*) where it was attached."""
    from rdkit import Chem
    mol = Chem.MolFromSmiles(smiles)
    cuts = _cut_bonds(mol, atom_groups)
    pieces = Chem.FragmentOnBonds(mol, cuts, addDummies=True, dummyLabels=[(0, 0)] * len(cuts)) if cuts else mol
    piece_atoms = Chem.GetMolFrags(pieces, asMols=False, sanitizeFrags=False)
    owner = {a: p for p, atoms in enumerate(piece_atoms) for a in atoms}
    out = []
    for g in atom_groups:
        atoms = piece_atoms[owner[int(g[0])]]
        smi = Chem.MolFragmentToSmiles(pieces, atomsToUse=list(atoms), canonical=False, kekuleSmiles=False)
        m = Chem.MolFromSmiles(smi, sanitize=False)
        m.UpdatePropertyCache(strict=False)
        Chem.SanitizeMol(m, Chem.SanitizeFlags.SANITIZE_ALL ^ Chem.SanitizeFlags.SANITIZE_KEKULIZE
                         ^ Chem.SanitizeFlags.SANITIZE_SETAROMATICITY, catchErrors=True)
        out.append(m)
    return out


def draw_fragment_tile(frag_mol, color, size=(220, 200), legend=""):
    """One fragment at a fixed bond length (so a single atom isn't blown up to fill the tile)."""
    from rdkit.Chem.Draw import rdMolDraw2D
    atoms = [a.GetIdx() for a in frag_mol.GetAtoms() if a.GetAtomicNum() != 0]
    bonds = [b.GetIdx() for b in frag_mol.GetBonds()
             if b.GetBeginAtom().GetAtomicNum() != 0 and b.GetEndAtom().GetAtomicNum() != 0]
    d = _drawer(size, legend_size=13)
    d.drawOptions().padding = 0.15
    d.drawOptions().fixedBondLength = 28
    rdMolDraw2D.PrepareAndDrawMolecule(
        d, frag_mol, legend=legend, kekulize=False,
        highlightAtoms=atoms, highlightAtomColors={a: color for a in atoms},
        highlightBonds=bonds, highlightBondColors={b: color for b in bonds},
        highlightAtomRadii={a: 0.4 for a in atoms})
    return _png(d)


def _images(pngs):
    import io
    from PIL import Image
    return [Image.open(io.BytesIO(p)).convert("RGB") for p in pngs]


def _font(size):
    from PIL import ImageFont
    for path in ("DejaVuSans.ttf", "/usr/share/fonts/truetype/dejavu/DejaVuSans.ttf"):
        try:
            return ImageFont.truetype(path, size)
        except OSError:
            pass
    try:
        return ImageFont.load_default(size=size)
    except TypeError:
        return ImageFont.load_default()


def _caption(img, lines, size=14, bold_first=True, pad=6):
    """Image with text lines underneath (readable size, unlike RDKit legends that shrink)."""
    from PIL import Image, ImageDraw
    f, fb = _font(size), _font(size + 1)
    line_h = size + 5
    out = Image.new("RGB", (img.width, img.height + pad + line_h * len(lines)), "white")
    out.paste(img, (0, 0))
    draw = ImageDraw.Draw(out)
    for i, line in enumerate(lines):
        font = fb if (i == 0 and bold_first) else f
        w = draw.textlength(line, font=font)
        draw.text(((img.width - w) / 2, img.height + pad + i * line_h), line, fill="black", font=font)
    return out


def _stack_rows(rows, pad=10, bg="white"):
    from PIL import Image
    width = max(sum(im.width for im in r) + pad * (len(r) - 1) for r in rows)
    height = sum(max(im.height for im in r) for r in rows) + pad * (len(rows) - 1)
    canvas = Image.new("RGB", (width, height), bg)
    y = 0
    for r in rows:
        x = 0
        for im in r:
            canvas.paste(im, (x, y))
            x += im.width + pad
        y += max(im.height for im in r) + pad
    return canvas


def explanation_grid(records, atom_spec=None, device=None, n=4, out_path="figures/cam_examples.png"):
    """Rows: molecules. Columns: atom Grad-CAM, fragment Grad-CAM, GSPELL YES fragments.
    Atom Grad-CAM comes from the stored atom_gradcam per-atom scores if atom_spec is None."""
    rows = []
    for _, r in records.head(n).iterrows():
        smi, groups = r["smiles"], r["atom_groups"]
        panels = []
        if atom_spec is not None:
            a = atom_gradcam_scores(atom_spec, smi, device).numpy()
            panels.append(draw_fragment_map(smi, groups, atom_scores=a, legend="Grad-CAM, atom GIN"))
        panels.append(draw_fragment_map(smi, groups, frag_scores=r["frag_gradcam_scores"], legend="Grad-CAM, fragment GIN"))
        panels.append(draw_fragment_map(smi, groups, keep=r["gspell_keep"], legend="GSPELL: YES fragments"))
        rows.append(_images(panels))
    grid = _stack_rows(rows)
    os.makedirs(os.path.dirname(out_path) or ".", exist_ok=True)
    grid.save(out_path)
    return grid


SCORE_LABELS = {"shapley": "Shapley", "frag_gradcam": "fCAM", "atom_gradcam": "aCAM", "attention": "attn", "gnnexp": "GNNExp"}


def fragment_panels(records, n=4, color_by="frag_gradcam", out_path="figures/fragment_panels.png",
                    max_per_row=6):
    """Per molecule: the molecule with numbered fragments, then each fragment on its own,
    colored by `color_by` scores and captioned with GSPELL's verdict and every method's score."""
    import textwrap
    from rdkit import Chem
    rows = []
    for _, r in records.head(n).iterrows():
        smi, groups = r["smiles"], r["atom_groups"]
        key = color_by if isinstance(r.get(f"{color_by}_scores"), (list, tuple)) else "frag_gradcam"
        scores = np.asarray(r[f"{key}_scores"], dtype=float)
        mol_png = draw_fragment_map(smi, groups, frag_scores=scores, size=(380, 300),
                                    legend=f"GIN score {r['gin_score']:.2f} | colored by {SCORE_LABELS.get(key, key)}")
        tiles = []
        cols = _colors(scores)
        keep = set(r["gspell_keep"])
        for f, (fm, c) in enumerate(zip(fragment_mols(smi, groups), cols)):
            vals = [f"{SCORE_LABELS[k]} {r[f'{k}_scores'][f]:+.2f}" for k in SCORE_LABELS
                    if isinstance(r.get(f"{k}_scores"), (list, tuple))]
            label = r["fragments"][f].split(" (", 1)[-1].rstrip(")") if " (" in r["fragments"][f] else ""
            ci = "counterion" in r and r["counterion"] and r["counterion"][f]
            flip = isinstance(r.get("removal_flips"), (list, tuple)) and r["removal_flips"][f]
            head = f"F{f + 1}{' [ion]' if ci else ''}  GSPELL {'YES' if f in keep else 'no'}{'  removal flips' if flip else ''}"
            lines = [head, textwrap.shorten(label, 36, placeholder="...")] + ["   ".join(vals[i:i + 2]) for i in range(0, len(vals), 2)]
            tiles.append(_caption(_images([draw_fragment_tile(fm, c, size=(230, 190))])[0], lines))
        imgs = _images([mol_png]) + tiles
        # wrap long fragment lists onto continuation rows
        rows.append(imgs[:1 + max_per_row])
        for k in range(1 + max_per_row, len(imgs), max_per_row):
            from PIL import Image
            rows.append([Image.new("RGB", imgs[0].size, "white")] + imgs[k:k + max_per_row])
    grid = _stack_rows(rows, pad=12)
    os.makedirs(os.path.dirname(out_path) or ".", exist_ok=True)
    grid.save(out_path)
    return grid


# ----------------------------------------------------------------------------- agreement

def agreement_table(records, methods=("gnnexp", "frag_gradcam", "attention", "atom_gradcam")):
    """Per method: mean Jaccard of its size-matched top-k with GSPELL's YES set, and mean
    Spearman correlation of its fragment scores with fragment Grad-CAM."""
    import pandas as pd
    from scipy.stats import spearmanr
    rows = []
    for m in methods:
        if f"{m}_scores" not in records:
            continue
        jac, rho = [], []
        for _, r in records.iterrows():
            a = set(r["gspell_keep"])
            k = max(1, len(a))
            b = set(np.argsort(-np.asarray(r[f"{m}_scores"]))[:k].tolist())
            if a:
                jac.append(len(a & b) / len(a | b))
            if m != "frag_gradcam" and r["n_fragments"] >= 3:
                rho.append(spearmanr(r[f"{m}_scores"], r["frag_gradcam_scores"]).correlation)
        rows.append({"method": m, "jaccard_with_GSPELL": np.nanmean(jac) if jac else np.nan,
                     "spearman_with_frag_gradcam": np.nanmean(rho) if rho else np.nan})
    return pd.DataFrame(rows)


def counterion_usage(records, methods=("gspell", "llm", "gnnexp", "gnnexp_k", "frag_gradcam", "attention", "atom_gradcam")):
    """Ionic mode only. Over molecules that have counter-ion nodes: how often each method's
    explanation includes a counter-ion, and the share of counter-ion nodes it selects.
    Compare with the base rate (share of all fragments that are counter-ions)."""
    import pandas as pd
    salts = records[records["counterion"].apply(any)]
    if len(salts) == 0:
        return pd.DataFrame()
    base = np.mean([sum(c) / len(c) for c in salts["counterion"]])
    rows = []
    for m in methods:
        if f"{m}_keep" not in salts:
            continue
        incl, share = [], []
        for _, r in salts.iterrows():
            ci = {i for i, c in enumerate(r["counterion"]) if c}
            keep = set(r[f"{m}_keep"])
            incl.append(bool(keep & ci))
            share.append(len(keep & ci) / len(ci))
        rows.append({"method": m, "molecules_with_counterions": len(salts),
                     "pct_explanations_including_counterion": 100 * np.mean(incl),
                     "pct_counterion_nodes_selected": 100 * np.mean(share),
                     "base_rate_counterion_fraction_%": 100 * base})
    return pd.DataFrame(rows)

In [ ]:
%%writefile viz_mol.py
"""Model diagnostics and visual summaries.

  predict_all            test-set scores, labels, molecule and fragment embeddings
  confusion_figure       classification: confusion matrix (counts and row-normalized) + metrics
  parity_figure          regression: predicted vs. measured, with RMSE / MAE / R^2
                         (the regression counterpart of a confusion matrix)
  embedding_figure       t-SNE and UMAP of molecule embeddings, colored by label and by correctness
  fragment_embedding_figure  t-SNE / UMAP of fragment embeddings, colored by functional group
  attention_by_group     mean readout attention per functional group (bar chart)
  attention_maps         molecules drawn with readout-attention weights per fragment
  explanation_heatmap    fragments x methods heatmap for one molecule (GSPELL, Grad-CAM, attention, ...)
  alignment_figure       UMAP of projected soft prompts next to the LLM text embeddings they align to
"""

import os
from collections import Counter

import matplotlib
import matplotlib.pyplot as plt
import numpy as np
import torch
from torch_geometric.loader import DataLoader
from torch_geometric.utils import softmax as segment_softmax

plt.rcParams.update({"figure.dpi": 120, "savefig.dpi": 200, "font.size": 9})


def _save(fig, path):
    os.makedirs(os.path.dirname(path) or ".", exist_ok=True)
    fig.savefig(path, bbox_inches="tight")
    fig.savefig(os.path.splitext(path)[0] + ".pdf", bbox_inches="tight")


# ----------------------------------------------------------------------------- predictions

@torch.no_grad()
def predict_all(spec, graphs, device, batch_size=256):
    """Scores (probability or value), labels, molecule embeddings, fragment embeddings,
    fragment readout attention and fragment texts for a list of graphs."""
    gin = spec.gin.eval()
    scores, ys, G, H, A, texts = [], [], [], [], [], []
    for b in DataLoader(graphs, batch_size=batch_size):
        b = b.to(device)
        h, g = gin.embed(b)
        scores.append(spec.score(gin.head(g).squeeze(-1)).cpu())
        ys.append(b.y.cpu())
        G.append(g.cpu()); H.append(h.cpu())
        A.append(segment_softmax(gin.readout.gate_nn(h).view(-1), b.batch).cpu())
        texts += [t for ts in b.fragment_text for t in ts]
    return {"score": torch.cat(scores).numpy(), "y": torch.cat(ys).numpy(),
            "graph_emb": torch.cat(G).numpy(), "frag_emb": torch.cat(H).numpy(),
            "frag_attention": torch.cat(A).numpy(), "frag_text": texts}


def fragment_group(text):
    """First functional-group label from node text 'SMILES (label, label)'."""
    if " (" not in text:
        return "other"
    inner = text.split(" (", 1)[1].rstrip(")")
    if inner.startswith("counter-ion"):
        return "counter-ion"
    return inner.split(",")[0].strip()


# ----------------------------------------------------------------------------- classification / regression

def confusion_figure(pred, title, path, threshold=0.5, labels=("negative", "positive")):
    from sklearn.metrics import (accuracy_score, balanced_accuracy_score, confusion_matrix,
                                 f1_score, precision_score, recall_score, roc_auc_score)
    y, p = pred["y"].astype(int), pred["score"]
    yhat = (p >= threshold).astype(int)
    cm = confusion_matrix(y, yhat, labels=[0, 1])
    cmn = cm / cm.sum(axis=1, keepdims=True).clip(min=1)
    metrics = {"ROC-AUC": roc_auc_score(y, p), "accuracy": accuracy_score(y, yhat),
               "balanced acc.": balanced_accuracy_score(y, yhat), "precision": precision_score(y, yhat, zero_division=0),
               "recall": recall_score(y, yhat, zero_division=0), "F1": f1_score(y, yhat, zero_division=0)}
    fig, axes = plt.subplots(1, 2, figsize=(7.2, 3.2))
    for ax, mat, fmt, sub in [(axes[0], cm, "d", "counts"), (axes[1], cmn, ".2f", "row-normalized (recall)")]:
        ax.imshow(cmn, cmap="Blues", vmin=0, vmax=1)
        for i in range(2):
            for j in range(2):
                ax.text(j, i, format(mat[i, j], fmt), ha="center", va="center",
                        color="white" if cmn[i, j] > 0.6 else "black", fontsize=11)
        ax.set_xticks([0, 1], labels); ax.set_yticks([0, 1], labels)
        ax.set_xlabel("predicted"); ax.set_ylabel("true"); ax.set_title(sub, fontsize=9)
    fig.suptitle(title + "\n" + "  ".join(f"{k} {v:.3f}" for k, v in metrics.items()), fontsize=9)
    fig.tight_layout()
    _save(fig, path)
    return fig, cm, metrics


def parity_figure(pred, title, path, threshold=None):
    y, p = pred["y"], pred["score"]
    rmse = float(np.sqrt(((p - y) ** 2).mean())); mae = float(np.abs(p - y).mean())
    r2 = 1 - ((p - y) ** 2).sum() / ((y - y.mean()) ** 2).sum()
    fig, ax = plt.subplots(figsize=(3.6, 3.4))
    lo, hi = min(y.min(), p.min()), max(y.max(), p.max())
    ax.plot([lo, hi], [lo, hi], "--", color="gray", lw=1)
    ax.scatter(y, p, s=10, alpha=0.6, edgecolors="none")
    if threshold is not None:   # the cut used to turn ESOL into classes for explanation
        ax.axvline(threshold, color="#c0392b", lw=0.8, alpha=0.6)
        ax.axhline(threshold, color="#c0392b", lw=0.8, alpha=0.6)
    ax.set_xlabel("measured"); ax.set_ylabel("predicted")
    ax.set_title(f"{title}\nRMSE {rmse:.3f}  MAE {mae:.3f}  $R^2$ {r2:.3f}", fontsize=9)
    fig.tight_layout()
    _save(fig, path)
    return fig, {"RMSE": rmse, "MAE": mae, "R2": float(r2)}


# ----------------------------------------------------------------------------- embeddings

def _reduce(X, method, seed=0):
    from sklearn.preprocessing import StandardScaler
    X = StandardScaler().fit_transform(X)
    if method == "tsne":
        from sklearn.manifold import TSNE
        return TSNE(n_components=2, perplexity=min(30, max(5, len(X) // 4)), init="pca",
                    random_state=seed).fit_transform(X)
    import umap   # pip install umap-learn
    return umap.UMAP(n_neighbors=min(15, len(X) - 1), min_dist=0.1, random_state=seed).fit_transform(X)


def available_methods():
    try:
        import umap  # noqa: F401
        return ["tsne", "umap"]
    except ImportError:
        return ["tsne"]


def embedding_figure(pred, task, title, path, threshold=0.5, methods=None):
    """Rows: t-SNE / UMAP. Columns: colored by true label (or measured value), and by
    whether the prediction is correct (classification) or by absolute error (regression)."""
    methods = methods or available_methods()
    fig, axes = plt.subplots(len(methods), 2, figsize=(7.4, 3.3 * len(methods)), squeeze=False)
    y, p = pred["y"], pred["score"]
    for r, m in enumerate(methods):
        Z = _reduce(pred["graph_emb"], m)
        ax = axes[r, 0]
        if task == "classification":
            for c, col, lab in [(0, "#3b75af", "label 0"), (1, "#d1495b", "label 1")]:
                ax.scatter(*Z[y == c].T, s=8, alpha=0.7, color=col, label=lab, edgecolors="none")
            ax.legend(frameon=False, fontsize=7, markerscale=2)
        else:
            sc = ax.scatter(*Z.T, s=8, c=y, cmap="viridis", edgecolors="none")
            fig.colorbar(sc, ax=ax, fraction=0.046, label="measured")
        ax.set_title(f"{m.upper()} — true label", fontsize=9)
        ax = axes[r, 1]
        if task == "classification":
            ok = (p >= threshold) == (y >= 0.5)
            ax.scatter(*Z[ok].T, s=8, alpha=0.5, color="#9aa5b1", label="correct", edgecolors="none")
            ax.scatter(*Z[~ok].T, s=12, alpha=0.9, color="#e07b39", label="wrong", edgecolors="none")
            ax.legend(frameon=False, fontsize=7, markerscale=2)
            ax.set_title(f"{m.upper()} — prediction correct?", fontsize=9)
        else:
            sc = ax.scatter(*Z.T, s=8, c=np.abs(p - y), cmap="magma_r", edgecolors="none")
            fig.colorbar(sc, ax=ax, fraction=0.046, label="|error|")
            ax.set_title(f"{m.upper()} — absolute error", fontsize=9)
        for a in axes[r]:
            a.set_xticks([]); a.set_yticks([])
    fig.suptitle(title, fontsize=10)
    fig.tight_layout()
    _save(fig, path)
    return fig


def fragment_embedding_figure(pred, title, path, top=8, methods=None, max_points=6000, seed=0):
    """Fragment (node) embeddings colored by their main functional-group label."""
    methods = methods or available_methods()
    groups = np.array([fragment_group(t) for t in pred["frag_text"]])
    X = pred["frag_emb"]
    idx = np.arange(len(X))
    if len(idx) > max_points:
        idx = np.random.default_rng(seed).choice(idx, max_points, replace=False)
    X, groups = X[idx], groups[idx]
    common = [g for g, _ in Counter(groups).most_common(top)]
    cmap = matplotlib.colormaps["tab10"]
    fig, axes = plt.subplots(1, len(methods), figsize=(4.4 * len(methods), 3.8), squeeze=False)
    for ax, m in zip(axes[0], methods):
        Z = _reduce(X, m, seed)
        rest = ~np.isin(groups, common)
        ax.scatter(*Z[rest].T, s=4, color="#d0d0d0", edgecolors="none", label="other groups")
        for k, g in enumerate(common):
            sel = groups == g
            ax.scatter(*Z[sel].T, s=5, color=cmap(k), edgecolors="none", label=g)
        ax.set_title(m.upper(), fontsize=9); ax.set_xticks([]); ax.set_yticks([])
    axes[0, -1].legend(frameon=False, fontsize=7, markerscale=3, loc="center left", bbox_to_anchor=(1, 0.5))
    fig.suptitle(title, fontsize=10)
    fig.tight_layout()
    _save(fig, path)
    return fig


# ----------------------------------------------------------------------------- attention

def attention_by_group(pred, title, path, top=12, min_count=10):
    """Mean readout attention per functional group, relative to uniform attention within each
    molecule (1.0 = the fragment gets its fair share; > 1 = attended more than average)."""
    import pandas as pd
    # attention relative to 1/n_fragments of its molecule needs molecule sizes: recover from
    # attention sums (each molecule's weights sum to 1) by walking the flat list.
    rel, i = [], 0
    att = pred["frag_attention"]
    sizes = pred.get("n_frag")
    if sizes is None:
        raise ValueError("pass pred from predict_all_with_sizes")
    for n in sizes:
        rel += list(att[i:i + n] * n)
        i += n
    df = pd.DataFrame({"group": [fragment_group(t) for t in pred["frag_text"]], "rel_attention": rel})
    stats = df.groupby("group")["rel_attention"].agg(["mean", "sem", "count"])
    stats = stats[stats["count"] >= min_count].sort_values("mean", ascending=False).head(top)
    fig, ax = plt.subplots(figsize=(4.8, 0.28 * len(stats) + 1.0))
    ax.barh(stats.index[::-1], stats["mean"][::-1], xerr=stats["sem"][::-1], color="#4c78a8", alpha=0.85)
    ax.axvline(1.0, color="gray", ls="--", lw=1)
    ax.set_xlabel("readout attention relative to uniform")
    ax.set_title(title, fontsize=9)
    fig.tight_layout()
    _save(fig, path)
    return fig, stats


def predict_all_with_sizes(spec, graphs, device):
    pred = predict_all(spec, graphs, device)
    pred["n_frag"] = [int(g.num_nodes) for g in graphs]
    pred["smiles"] = [g.smiles for g in graphs]
    return pred


def attention_maps(spec, graphs, device, n=6, out_path="figures/attention_maps.png", seed=0, min_frag=3):
    """Molecules drawn with the GIN's readout attention on each fragment (darker = more attention)."""
    import random
    from cam_mol import _caption, _images, _stack_rows, attention, draw_fragment_map
    from gspell_mol import canonical_graph
    pool = [g for g in graphs if g.num_nodes >= min_frag]
    random.Random(seed).shuffle(pool)
    tiles = []
    for g in pool[:n]:
        d = canonical_graph(g.cpu())
        a = attention(spec, d, device).numpy()
        png = draw_fragment_map(d.smiles, d.atom_groups, frag_scores=a - 1.0 / len(a), size=(320, 260))
        top = int(np.argmax(a))
        tiles.append(_caption(_images([png])[0],
                              [f"y = {float(d.y):.2f}   max attention F{top + 1} ({a[top]:.2f})"], size=13))
    rows = [tiles[i:i + 3] for i in range(0, len(tiles), 3)]
    grid = _stack_rows(rows, pad=12)
    os.makedirs(os.path.dirname(out_path) or ".", exist_ok=True)
    grid.save(out_path)
    return grid


# ----------------------------------------------------------------------------- explanations

def explanation_heatmap(rec, path, title=""):
    """One molecule: fragments (rows) x explainers (columns). Each column is scaled to [-1, 1]
    within the molecule so methods with different units can be compared by rank and sign."""
    cols = [("GSPELL", None), ("Shapley (reference)", "shapley"), ("fragment Grad-CAM", "frag_gradcam"),
            ("atom Grad-CAM", "atom_gradcam"), ("readout attention", "attention"), ("GNNExplainer", "gnnexp")]
    cols = [(n, k) for n, k in cols if k is None or isinstance(rec.get(f"{k}_scores"), (list, tuple))]
    n = rec["n_fragments"]
    M = np.zeros((n, len(cols)))
    for j, (_, k) in enumerate(cols):
        if k is None:
            M[:, j] = [1.0 if i in rec["gspell_keep"] else 0.0 for i in range(n)]
        else:
            v = np.asarray(rec[f"{k}_scores"], dtype=float)
            if k in ("attention", "gnnexp"):
                v = v - v.mean()
            M[:, j] = v / max(np.abs(v).max(), 1e-8)
    flips = rec.get("removal_flips") if isinstance(rec.get("removal_flips"), (list, tuple)) else [False] * n
    ions = rec.get("counterion") if isinstance(rec.get("counterion"), (list, tuple)) else [False] * n
    labels = [f"F{i + 1} " + t.split(" (")[0][:22] + (" [ion]" if ions[i] else "") + (" (removal flips)" if flips[i] else "")
              for i, t in enumerate(rec["fragments"])]
    fig, ax = plt.subplots(figsize=(1.0 * len(cols) + 2.6, 0.38 * n + 1.3))
    im = ax.imshow(M, cmap="RdBu_r", vmin=-1, vmax=1, aspect="auto")
    ax.set_xticks(range(len(cols)), [c for c, _ in cols], rotation=30, ha="right")
    ax.set_yticks(range(n), labels, fontsize=8)
    for i in range(n):
        for j in range(len(cols)):
            ax.text(j, i, f"{M[i, j]:+.2f}" if j else ("YES" if M[i, j] else "no"), ha="center", va="center", fontsize=7,
                    color="white" if abs(M[i, j]) > 0.6 else "black")
    fig.colorbar(im, ax=ax, fraction=0.04, label="scaled score (red supports prediction)")
    ax.set_title(title or rec["smiles"][:60], fontsize=9)
    fig.tight_layout()
    _save(fig, path)
    return fig


@torch.no_grad()
def alignment_figure(spec, proj, graphs, text_emb_fn, device, title, path, n_frag=1500, seed=0):
    """UMAP (or t-SNE) of projected fragment soft prompts (mean over k tokens) together with the
    LLM text embeddings of the same fragments, the two sides GSPELL's context loss aligns."""
    from gspell_mol import collect_embeddings
    Hn, Tn, _, _ = collect_embeddings(spec.gin, graphs, device)
    rng = np.random.default_rng(seed)
    idx = rng.choice(len(Tn), min(n_frag, len(Tn)), replace=False)
    Z = torch.nn.functional.normalize(proj(Hn[idx].to(device)).mean(1).float(), dim=-1).cpu().numpy()
    T = torch.nn.functional.normalize(text_emb_fn([Tn[i] for i in idx]).float(), dim=-1).numpy()
    cos = (Z * T).sum(1)
    method = available_methods()[-1]
    E = _reduce(np.concatenate([Z, T]), method, seed)
    groups = np.array([fragment_group(Tn[i]) for i in idx])
    common = [g for g, _ in Counter(groups).most_common(6)]
    cmap = matplotlib.colormaps["tab10"]
    fig, axes = plt.subplots(1, 2, figsize=(9.0, 3.8))
    ax = axes[0]
    ax.scatter(*E[:len(idx)].T, s=5, color="#e07b39", alpha=0.6, label="soft prompts (projected GIN)", edgecolors="none")
    ax.scatter(*E[len(idx):].T, s=5, color="#4c78a8", alpha=0.6, label="LLM text embeddings", edgecolors="none")
    ax.legend(frameon=False, fontsize=7, markerscale=3); ax.set_title(f"{method.upper()}: both spaces", fontsize=9)
    ax = axes[1]
    for k, g in enumerate(common):
        sel = groups == g
        ax.scatter(*E[:len(idx)][sel].T, s=6, color=cmap(k), label=g, edgecolors="none")
    ax.set_title("soft prompts by functional group", fontsize=9)
    ax.legend(frameon=False, fontsize=7, markerscale=3, loc="center left", bbox_to_anchor=(1, 0.5))
    for a in axes:
        a.set_xticks([]); a.set_yticks([])
    fig.suptitle(f"{title}  (mean cosine soft prompt vs. its text: {cos.mean():.3f})", fontsize=10)
    fig.tight_layout()
    _save(fig, path)
    return fig, float(cos.mean())


# ----------------------------------------------------------------------------- faithfulness figures

def faithfulness_figure(deletion_tables, agreement_tables, title, path):
    """Left: fragments removed (share) before the class flips, per explainer and dataset (lower =
    the ranking finds the decisive fragments sooner). Right: GSPELL vs. the text-only LLM on
    agreement with Shapley (AUROC: do YES fragments carry higher Shapley values?) and recall of
    the fragments whose removal alone flips the prediction."""
    import pandas as pd
    names = list(deletion_tables)
    methods = ["Shapley (reference)", "GSPELL (graph-level)", "LLM text-only (no projector)",
               "Grad-CAM, fragment GIN", "Grad-CAM, atom GIN -> fragments", "Readout attention",
               "GNNExplainer", "Random order"]
    methods = [m for m in methods if any(m in deletion_tables[n].index for n in names)]
    fig, axes = plt.subplots(1, 2, figsize=(11, 3.8), gridspec_kw={"width_ratios": [1.3, 1]})
    ax = axes[0]
    h = 0.8 / max(1, len(names))
    cmap = matplotlib.colormaps["tab10"]
    y = np.arange(len(methods))[::-1]          # first method at the top
    for j, n in enumerate(names):
        t = deletion_tables[n]
        vals = [t.loc[m, "steps_to_flip"] if m in t.index else np.nan for m in methods]
        ax.barh(y + 0.4 - h / 2 - j * h, vals, h, label=n, color=cmap(j))
    ax.set_yticks(y, methods, fontsize=7.5)
    ax.set_xlabel("share of fragments removed before the class flips (lower = better)", fontsize=8)
    ax.legend(frameon=False, fontsize=7, loc="upper center", bbox_to_anchor=(0.5, -0.16), ncol=len(names))
    ax = axes[1]
    rows = []
    for n in names:
        a = agreement_tables[n]
        for m in ["GSPELL (graph-level)", "LLM text-only (no projector)"]:
            if m in a.index:
                rows.append({"dataset": n, "method": "GSPELL" if m.startswith("GSPELL") else "text-only LLM",
                             "AUROC vs Shapley": a.loc[m].get("auroc_phi", np.nan),
                             "recall of decisive": a.loc[m].get("recall_decisive", np.nan)})
    df = pd.DataFrame(rows)
    if len(df):
        metrics = ["AUROC vs Shapley", "recall of decisive"]
        groups = [(d, m) for d in names for m in ["GSPELL", "text-only LLM"]]
        x = np.arange(len(names))
        for k, (meth, col) in enumerate([("GSPELL", "#e07b39"), ("text-only LLM", "#9aa5b1")]):
            for mi, metric in enumerate(metrics):
                vals = [df[(df.dataset == d) & (df.method == meth)][metric].mean() for d in names]
                ax.bar(x + (mi * 2 + k) * 0.2 - 0.3, vals, 0.18, color=col, alpha=1.0 if mi == 0 else 0.55,
                       label=f"{meth}: {metric}")
        ax.axhline(0.5, color="gray", ls="--", lw=0.8)
        ax.set_xticks(x, names); ax.set_ylim(0, 1.05)
        ax.legend(frameon=False, fontsize=6.5, loc="upper left", bbox_to_anchor=(1, 1))
        ax.set_title("LLM support decisions vs. model references", fontsize=9)
    fig.suptitle(title, fontsize=10)
    fig.tight_layout()
    _save(fig, path)
    return fig


def qa_figure(qa_tables, title, path):
    """QA accuracy with and without the GIN's soft prompts, per question type, against chance."""
    names = list(qa_tables)
    fig, axes = plt.subplots(1, len(names), figsize=(3.4 * len(names), 3.2), squeeze=False, sharey=True)
    for ax, n in zip(axes[0], names):
        t = qa_tables[n]
        t = t.drop(index="all", errors="ignore")
        x = np.arange(len(t))
        ax.bar(x - 0.2, t["with_soft_prompts"], 0.38, color="#e07b39", label="with soft prompts")
        ax.bar(x + 0.2, t["text_only"], 0.38, color="#9aa5b1", label="text only")
        ax.scatter(x, t["chance"], marker="_", s=400, color="black", label="chance", zorder=3)
        ax.set_xticks(x, [i.replace("_", "\n") for i in t.index], fontsize=7)
        ax.set_ylim(0, 1.05); ax.set_title(n, fontsize=9)
    axes[0, 0].set_ylabel("accuracy")
    axes[0, -1].legend(frameon=False, fontsize=7, loc="upper left", bbox_to_anchor=(1, 1))
    fig.suptitle(title, fontsize=10)
    fig.tight_layout()
    _save(fig, path)
    return fig

In [ ]:
%%writefile faithfulness.py
"""Model-derived references for judging explanations: exact fragment Shapley values,
removal counterfactuals, deletion curves, agreement metrics, and a QA faithfulness test.

The game (Roth, JCIM 2026; computed on the fragment GIN)
  players  = the molecule's fragments (n <= 12, so all 2^n coalitions are enumerated)
  v(S)     = the GIN's output on the subgraph induced by the fragments in S, ORIENTED toward the
             predicted class and centred on the decision boundary:
                 v(S) = s * (raw(S) - c)
             raw = logit (classification) or standardized value (regression); c = 0 for logits,
             (threshold - mu) / sd for regression; s = +1 if the full molecule is predicted
             positive / above the threshold, else -1. So v(S) > 0 means "the subgraph S still
             gets the same class as the whole molecule", and phi_i > 0 means fragment i pushes
             toward the prediction.
  v(empty) = the prediction head applied to a zero molecule embedding (the GIN's attention
             readout of an empty set), i.e. the model's "no evidence" output.

Exactness is checked by the efficiency axiom: sum_i phi_i = v(all) - v(empty).

Removal counterfactual for fragment i: does v(all minus i) < 0, i.e. does removing i alone flip
the class? Taken from the same table, no extra model calls.

Deletion curve for any ranking (Shapley, Grad-CAM, attention, GNNExplainer, GSPELL's YES set):
remove fragments in ranked order and read v from the table. Reported as (a) the share of
fragments removed before the class flips (lower = the ranking found the decisive fragments
sooner) and (b) the area under v / v(all) (lower = faster drop). Random orderings give chance.

QA faithfulness: questions whose answers come from the model (Shapley, removal flips), asked to
the LLM with the GIN's soft prompts and without them. If accuracy is no better with soft
prompts, the LLM's rationale is driven by its chemistry priors, not by the model.
"""

import json
import os
import random
import re
from math import factorial

import numpy as np

# ----------------------------------------------------------------------------- the game


def popcounts(n):
    return np.array([bin(m).count("1") for m in range(2 ** n)])


def shapley_from_table(v, n):
    """Exact Shapley values from a table v[mask] over all 2^n coalitions (bit i = fragment i)."""
    v = np.asarray(v, dtype=float)
    masks = np.arange(2 ** n)
    size = popcounts(n)
    w = np.array([factorial(s) * factorial(n - s - 1) / factorial(n) for s in range(n)])
    phi = np.zeros(n)
    for i in range(n):
        bit = 1 << i
        without = masks[(masks & bit) == 0]
        phi[i] = np.sum(w[size[without]] * (v[without | bit] - v[without]))
    return phi


def full_mask(n):
    return (1 << n) - 1


def removal_flips(v, n):
    """For each fragment: does removing it alone flip the predicted class?"""
    full = full_mask(n)
    return [bool(v[full & ~(1 << i)] < 0) for i in range(n)]


def deletion_curve(v, n, order):
    """v after removing fragments in `order`, one at a time (length n + 1, starts at v(all))."""
    mask = full_mask(n)
    vals = [v[mask]]
    for i in order:
        mask &= ~(1 << int(i))
        vals.append(v[mask])
    return np.array(vals)


def deletion_metrics(v, n, order):
    curve = deletion_curve(v, n, order)
    flipped = np.where(curve[1:] < 0)[0]
    steps = (flipped[0] + 1) / n if len(flipped) else 1.0 + 1.0 / n   # never flips -> worst score
    base = curve[0] if curve[0] > 0 else 1.0
    auc = float(np.mean(curve / base))
    return {"steps_to_flip": float(steps), "deletion_auc": auc}


def random_deletion(v, n, n_perm=50, seed=0):
    rng = random.Random(seed)
    res = [deletion_metrics(v, n, rng.sample(range(n), n)) for _ in range(n_perm)]
    return {k: float(np.mean([r[k] for r in res])) for k in res[0]}


def ranking_from_set(keep, n, seed=0, n_perm=20):
    """Orders that put a set's members first (ties broken at random), for set-valued explainers."""
    rng = random.Random(seed)
    keep = list(keep)
    rest = [i for i in range(n) if i not in set(keep)]
    out = []
    for _ in range(n_perm):
        a, b = keep[:], rest[:]
        rng.shuffle(a); rng.shuffle(b)
        out.append(a + b)
    return out


# ----------------------------------------------------------------------------- the model side

def coalition_table(spec, data, device, batch_size=1024):
    """Oriented, centred values v[mask] for all 2^n coalitions of a fragment graph."""
    import torch
    from torch_geometric.data import Batch, Data
    from torch_geometric.utils import subgraph

    gin = spec.gin.eval()
    n = int(data.num_nodes)
    raw = np.zeros(2 ** n)
    with torch.no_grad():
        hidden = gin.head[0].in_features
        raw[0] = gin.head(torch.zeros(1, hidden, device=device)).view(-1).item()
        masks = list(range(1, 2 ** n))
        x, ei, ea = data.x.cpu(), data.edge_index.cpu(), data.edge_attr.cpu()
        for start in range(0, len(masks), batch_size):
            chunk = masks[start:start + batch_size]
            graphs = []
            for m in chunk:
                keep = torch.tensor([i for i in range(n) if m >> i & 1], dtype=torch.long)
                sei, sea = subgraph(keep, ei, ea, relabel_nodes=True, num_nodes=n)
                graphs.append(Data(x=x[keep], edge_index=sei, edge_attr=sea))
            b = Batch.from_data_list(graphs).to(device)
            raw[chunk] = gin(b).view(-1).cpu().numpy()
    if spec.task == "classification":
        c = 0.0
    else:
        c = (spec.threshold - spec.mu) / spec.sd
    s = 1.0 if raw[full_mask(n)] >= c else -1.0
    return s * (raw - c), s


def analyse_molecule(spec, data, device, rec=None, n_perm=50):
    """Shapley values, removal flips, and deletion metrics for one molecule; when `rec` (a row of
    the explanation records) is given, also scores every explainer in it against Shapley."""
    n = int(data.num_nodes)
    v, sign = coalition_table(spec, data, device)
    phi = shapley_from_table(v, n)
    out = {"smiles": data.smiles, "n_fragments": n, "v_full": float(v[full_mask(n)]), "v_empty": float(v[0]),
           "shapley_scores": phi.tolist(), "removal_flips": removal_flips(v, n),
           "efficiency_error": float(abs(phi.sum() - (v[full_mask(n)] - v[0]))),
           "pred_sign": sign}
    out["deletion"] = {"shapley": deletion_metrics(v, n, np.argsort(-phi)), "random": random_deletion(v, n, n_perm)}
    if rec is not None:
        out["agreement"] = agreement_with_shapley(rec, phi, out["removal_flips"])
        for key in SCORE_EXPLAINERS:
            if isinstance(rec.get(f"{key}_scores"), (list, tuple)):
                out["deletion"][key] = deletion_metrics(v, n, np.argsort(-np.asarray(rec[f"{key}_scores"])))
        for key in SET_EXPLAINERS:
            if isinstance(rec.get(f"{key}_keep"), (list, tuple)):
                res = [deletion_metrics(v, n, o) for o in ranking_from_set(rec[f"{key}_keep"], n)]
                out["deletion"][key] = {k: float(np.mean([r[k] for r in res])) for k in res[0]}
    return out


# ----------------------------------------------------------------------------- agreement

SCORE_EXPLAINERS = ["frag_gradcam", "atom_gradcam", "attention", "gnnexp"]
SET_EXPLAINERS = ["gspell", "llm"]


def _auroc(pos_scores, neg_scores):
    """P(random positive > random negative), ties count half."""
    if not len(pos_scores) or not len(neg_scores):
        return np.nan
    p, q = np.asarray(pos_scores)[:, None], np.asarray(neg_scores)[None, :]
    return float(((p > q) + 0.5 * (p == q)).mean())


def agreement_with_shapley(rec, phi, flips):
    """How well each explainer matches the model-derived references for one molecule."""
    from scipy.stats import spearmanr
    n = len(phi)
    phi = np.asarray(phi)
    support = {i for i in range(n) if phi[i] > 0}          # fragments pushing toward the prediction
    decisive = {i for i in range(n) if flips[i]}           # removing them alone flips the class
    res = {}
    for key in SET_EXPLAINERS:
        if not isinstance(rec.get(f"{key}_keep"), (list, tuple)):
            continue
        yes = set(rec[f"{key}_keep"])
        no = set(range(n)) - yes
        k = max(1, len(yes))
        topk = set(np.argsort(-phi)[:k].tolist())
        res[key] = {
            "auroc_phi": _auroc(phi[list(yes)], phi[list(no)]),           # do YES fragments have higher phi?
            "precision_support": len(yes & support) / len(yes) if yes else np.nan,
            "recall_support": len(yes & support) / len(support) if support else np.nan,
            "jaccard_topk": len(yes & topk) / len(yes | topk) if yes else 0.0,
            "recall_decisive": len(yes & decisive) / len(decisive) if decisive else np.nan,
            "size": len(yes),
        }
    for key in SCORE_EXPLAINERS:
        if not isinstance(rec.get(f"{key}_scores"), (list, tuple)):
            continue
        sc = np.asarray(rec[f"{key}_scores"], dtype=float)
        rho = spearmanr(sc, phi).correlation if n >= 3 and np.ptp(sc) > 1e-9 and np.ptp(phi) > 1e-9 else np.nan
        top1 = int(np.argmax(sc)) == int(np.argmax(phi))
        res[key] = {"spearman_phi": float(rho) if rho == rho else np.nan, "top1_match": float(top1),
                    "recall_decisive": (len(set(np.argsort(-sc)[:max(1, len(decisive))].tolist()) & decisive)
                                        / len(decisive)) if decisive else np.nan}
    return res


# ----------------------------------------------------------------------------- QA faithfulness

def make_questions(phi, flips, seed=0):
    """Questions whose correct answers come from the model. Fragment numbers are 1-based."""
    rng = random.Random(seed)
    n = len(phi)
    phi = np.asarray(phi)
    qs = []
    if n >= 2 and len(set(np.round(phi, 9))) > 1:
        qs.append({"type": "most_important", "answer": int(np.argmax(phi)) + 1, "chance": 1 / n,
                   "text": "Which single fragment contributes most to the GNN's prediction? "
                           "Answer with the fragment number only."})
    if n >= 2 and phi.min() < 0:
        qs.append({"type": "most_opposing", "answer": int(np.argmin(phi)) + 1, "chance": 1 / n,
                   "text": "Which single fragment works most against the GNN's prediction? "
                           "Answer with the fragment number only."})
    yes = [i for i in range(n) if flips[i]]
    no = [i for i in range(n) if not flips[i]]
    for pool, ans in ((yes, "YES"), (no, "NO")):            # one flipping and one non-flipping, when present
        if pool:
            i = rng.choice(pool)
            qs.append({"type": "removal_flip", "answer": ans, "chance": 0.5, "fragment": i + 1,
                       "text": f"If fragment {i + 1} were removed from the molecule, would the GNN's "
                               f"prediction change? Answer YES or NO only."})
    pairs = [(a, b) for a in range(n) for b in range(a + 1, n) if abs(phi[a] - phi[b]) > 1e-6]
    if pairs:
        a, b = rng.choice(pairs)
        if rng.random() < 0.5:
            a, b = b, a
        qs.append({"type": "pairwise", "answer": (a if phi[a] > phi[b] else b) + 1, "chance": 0.5,
                   "options": [a + 1, b + 1],
                   "text": f"Which fragment matters more for the GNN's prediction: fragment {a + 1} or "
                           f"fragment {b + 1}? Answer with the fragment number only."})
    return qs


def qa_prompt(data, label_text, question, with_soft=True):
    z = lambda key: f" <<Z_{key}>>" if with_soft else ""
    lines = [f"Target molecule:{z('G')}",
             f"GNN prediction: the molecule {label_text}.",
             "Fragments of the molecule:"]
    for i, t in enumerate(data.fragment_text):
        lines.append(f"Fragment {i + 1}: {t}{z(i)}")
    lines += ["", "Answer this question about the GNN's own reasoning, using the embeddings shown:", question]
    return "\n".join(lines)


def parse_answer(text, qtype, n, options=None):
    if qtype == "removal_flip":
        m = re.search(r"\b(YES|NO)\b", text, flags=re.I)
        return m.group(1).upper() if m else None
    for m in re.finditer(r"\d+", text):
        k = int(m.group())
        if 1 <= k <= n and (options is None or k in options):
            return k
    return None


def ask_questions(spec, proj, tok, llm, data, label_text, device, phi, flips, seed=0, max_new_tokens=12):
    """Ask each question with and without soft prompts. Returns a list of answered questions."""
    import torch
    from gspell_mol import generate
    n = int(data.num_nodes)
    with torch.no_grad():
        d = data.clone().to(device)
        d.batch = torch.zeros(n, dtype=torch.long, device=device)
        h, g = spec.gin.embed(d)
        zs = proj(torch.cat([h, g]))
    soft = {str(i): zs[i] for i in range(n)}
    soft["G"] = zs[n]
    out = []
    for q in make_questions(phi, flips, seed):
        row = dict(q)
        for cond, use_soft in (("soft", True), ("text_only", False)):
            raw = generate(tok, llm, qa_prompt(data, label_text, q["text"], use_soft),
                           soft if use_soft else None, max_new_tokens=max_new_tokens)
            ans = parse_answer(raw, q["type"], n, q.get("options"))
            row[f"{cond}_raw"] = raw.strip()[:80]
            row[f"{cond}_answer"] = ans
            row[f"{cond}_correct"] = float(ans == q["answer"])
        out.append(row)
    return out


# ----------------------------------------------------------------------------- summaries

METHOD_LABELS = {"gspell": "GSPELL (graph-level)", "llm": "LLM text-only (no projector)",
                 "frag_gradcam": "Grad-CAM, fragment GIN", "atom_gradcam": "Grad-CAM, atom GIN -> fragments",
                 "attention": "Readout attention", "gnnexp": "GNNExplainer", "shapley": "Shapley (reference)",
                 "random": "Random order"}


def summarize_agreement(rows):
    """rows: analyse_molecule outputs. Mean of each agreement metric per explainer."""
    import pandas as pd
    acc = {}
    for r in rows:
        for key, metrics in r.get("agreement", {}).items():
            for m, val in metrics.items():
                acc.setdefault((key, m), []).append(val)
    table = {}
    for (key, m), vals in acc.items():
        vals = [x for x in vals if x == x]          # drop NaN (metric undefined for that molecule)
        table.setdefault(METHOD_LABELS.get(key, key), {})[m] = float(np.mean(vals)) if vals else np.nan
    return pd.DataFrame(table).T


def summarize_deletion(rows):
    import pandas as pd
    acc = {}
    for r in rows:
        for key, metrics in r["deletion"].items():
            for m, val in metrics.items():
                acc.setdefault((key, m), []).append(val)
    table = {}
    for (key, m), vals in acc.items():
        table.setdefault(METHOD_LABELS.get(key, key), {})[m] = float(np.mean(vals))
    df = pd.DataFrame(table).T
    return df.sort_values("steps_to_flip") if "steps_to_flip" in df else df


def summarize_qa(qa_rows):
    import pandas as pd
    df = pd.DataFrame(qa_rows)
    if not len(df):
        return df
    g = df.groupby("type").agg(questions=("type", "size"), with_soft_prompts=("soft_correct", "mean"),
                               text_only=("text_only_correct", "mean"), chance=("chance", "mean"))
    total = pd.DataFrame([{"questions": len(df), "with_soft_prompts": df["soft_correct"].mean(),
                           "text_only": df["text_only_correct"].mean(), "chance": df["chance"].mean()}], index=["all"])
    g = pd.concat([g, total])
    g["grounding_gain"] = g["with_soft_prompts"] - g["text_only"]
    return g


def load_jsonl(path):
    if not path or not os.path.exists(path):
        return []
    with open(path) as f:
        return [json.loads(l) for l in f if l.strip()]


def append_jsonl(path, row):
    with open(path, "a") as f:
        f.write(json.dumps(row, default=float) + "\n")

## 3. Settings

`N_EVAL` is the main cost knob: molecules explained per dataset per mode. Raising it later extends the same molecule list and skips what's done.

In [ ]:
MODES      = ["strip", "ions"]        # covalent fragments only / covalent + ions
DATASETS   = ["BBBP", "BACE", "ESOL"]
SEEDS      = [0, 1, 2]                # GIN seeds for the results tables
GIN_SEED   = 0                        # the seed whose models get explained

RUN_DIAGNOSTICS = True                # stage C
RUN_EXPLAIN     = True                # stage D (needs the A100 and HF_TOKEN)
RUN_FIGURES     = True                # stage F

N_EVAL     = 30                       # molecules per dataset per mode
PAIRED     = True                     # same molecules in both modes
ONLY_SALTS = False                    # only molecules with counter-ions
TEXT_ONLY  = True                     # also run the LLM without soft prompts (doubles LLM time)
K_TOKENS   = 5
LLM_ID     = "meta-llama/Llama-3.1-8B-Instruct"
N_FIG      = 4                        # molecules per explanation figure
COLOR_BY   = "shapley"                # fragment panels: shapley, frag_gradcam, atom_gradcam, attention, gnnexp
RUN_QA     = True                     # QA faithfulness test (needs the LLM)

REBUILD_GRAPHS = False                # True rebuilds graphs even if they exist
RETRAIN        = False                # True retrains GINs even if checkpoints exist
DISCONNECT_WHEN_DONE = False          # True releases the A100 at the end to save compute units

PATHS = {   # mode -> (graph folder, checkpoint folder, file tag)
    "strip": ("frag_graphs", "checkpoints", ""),
    "ions":  ("frag_graphs_ions", "checkpoints_ions", "_ions"),
}
MODE_NAME = {"strip": "covalent only", "ions": "covalent + ions"}
for d in ("figures", "figures/diagnostics", "results"):
    os.makedirs(d, exist_ok=True)
LATEX = {}   # section -> LaTeX rows, written to results/latex_rows.txt at the end

In [ ]:
import importlib, fragment_graphs, frag_gin, prepare_data, gspell_mol, cam_mol, viz_mol, faithfulness
for m in (fragment_graphs, frag_gin, prepare_data, gspell_mol, cam_mol, viz_mol, faithfulness): importlib.reload(m)
import fragment_graphs as fg
from frag_gin import load_graphs, train_one, load_checkpoint, evaluate
from torch_geometric.loader import DataLoader
from gspell_mol import *
from cam_mol import *
from viz_mol import *
from faithfulness import (analyse_molecule, ask_questions, summarize_agreement, summarize_deletion,
                          summarize_qa, load_jsonl, append_jsonl)
TASKS = fg.DATASETS
METRIC = {n: "ROC-AUC" if TASKS[n][1] == "classification" else "RMSE" for n in DATASETS}

def stage(name):
    print(f"\n{'=' * 70}\n{name}   (elapsed {(time.time() - T0) / 60:.1f} min)\n{'=' * 70}")

# Stage A — Data

In [ ]:
stage("A. fragment graphs")
for name, path in CSVS.items():
    fg.CSV_COLUMNS[name] = prepare_data.detect_columns(name, path); fg.CSV_PATHS[name] = path

LV = fg.label_version()   # fragment-text version; graphs with older text are rebuilt (features are unchanged)
def labels_current(gdir, n):
    f = f"{gdir}/{n}_LABELS.txt"
    return os.path.exists(f) and open(f).read().strip() == LV

for mode in MODES:
    gdir = PATHS[mode][0]
    todo = [n for n in DATASETS if REBUILD_GRAPHS or not os.path.exists(f"{gdir}/{n}_test.pt") or not labels_current(gdir, n)]
    if todo:
        print(f"[{MODE_NAME[mode]}] building {todo}")
        fg.run(todo, build=True, out_dir=gdir, ionic=(mode == "ions"))
    else:
        print(f"[{MODE_NAME[mode]}] graphs exist")

stats = []
for mode in MODES:
    f = "fragment_stats_ions.csv" if mode == "ions" else "fragment_stats.csv"
    have = set(pd.read_csv(f)["dataset"]) if os.path.exists(f) else set()
    if not set(DATASETS) <= have:   # statistics missing: recompute (fast, no graph rebuild)
        fg.run([n for n in DATASETS if n not in have], build=False, ionic=(mode == "ions"))
    s = pd.read_csv(f); s["mode"] = MODE_NAME[mode]; stats.append(s[s.dataset.isin(DATASETS)])
stats = pd.concat(stats, ignore_index=True)
stats.to_csv("results/table1_fragment_stats.csv", index=False)
display(stats.round(2))
LATEX["Table 1: fragment statistics (Dataset & Mode & Task & #Mol & Frags & Max & % <=12 & % single)"] = [
    f"{r.dataset} & {r['mode']} & {r.task} & {r.molecules} & {r.mean_fragments:.1f} $\\pm$ {r.std_fragments:.1f} & "
    f"{r.max_fragments} & {r.pct_le_12:.1f} & {r.pct_single_fragment:.1f} \\\\" for _, r in stats.iterrows()]

In [ ]:
import matplotlib.pyplot as plt
fig, axes = plt.subplots(1, len(MODES), figsize=(4.2 * len(MODES), 2.8), squeeze=False, sharey=True)
for ax, mode in zip(axes[0], MODES):
    for name in DATASETS:
        counts = [d.num_nodes for d in sum(load_graphs(name, PATHS[mode][0]), [])]
        ax.hist(counts, bins=range(1, max(counts) + 2), histtype="step", linewidth=1.5, label=name, density=True)
    ax.axvline(12.5, color="gray", ls="--", lw=1)
    ax.set_title(MODE_NAME[mode], fontsize=9); ax.set_xlabel("BRICS fragments per molecule")
axes[0, 0].set_ylabel("fraction of molecules"); axes[0, 0].legend(frameon=False, fontsize=8)
fig.tight_layout(); fig.savefig("figures/fragment_hist_modes.pdf"); fig.savefig("figures/fragment_hist_modes.png", dpi=200)
display(fig); plt.close(fig)

In [ ]:
if "ions" in MODES:
    rows = []
    for name in [n for n in DATASETS if TASKS[n][1] == "classification"]:
        gs = sum(load_graphs(name, PATHS["ions"][0]), [])
        df = pd.DataFrame({"y": [int(d.y) for d in gs], "salt": [bool(d.counterion.any()) for d in gs]})
        for y, g in df.groupby("y"):
            rows.append({"dataset": name, "label": y, "molecules": len(g), "% salt forms": 100 * g["salt"].mean()})
    salts = pd.DataFrame(rows); salts.to_csv("results/salts_by_label.csv", index=False)
    print("Salt forms by label (a large gap means the counter-ion could act as a shortcut):"); display(salts.round(1))

## Functional-group vocabulary mined from the data

Ertl's list-free algorithm finds every functional group in the training molecules; the table shows the most common ones and how often the curated list already names them. Low percentages point to groups worth adding to the curated list. The second table shows how many fragments needed the list-free fallback label (`group ...`).

In [ ]:
from functional_groups import mine_groups
vocab = []
for name in DATASETS:
    smiles = [d.smiles for d in load_graphs(name, PATHS[MODES[0]][0])[0]]
    v = mine_groups(smiles, top=25); v.insert(0, "dataset", name); vocab.append(v)
vocab = pd.concat(vocab); vocab.to_csv("results/fg_vocabulary.csv", index=False)
display(vocab.round(1))
cov = []
for mode in MODES:
    for name in DATASETS:
        texts = [t for d in sum(load_graphs(name, PATHS[mode][0]), []) for t in d.fragment_text]
        cov.append({"dataset": name, "mode": MODE_NAME[mode], "fragments": len(texts),
                    "% with list-free label": 100 * np.mean(["(group " in t or ", group " in t for t in texts]),
                    "% labelled 'other'": 100 * np.mean(["(other)" in t for t in texts])})
display(pd.DataFrame(cov).round(1))

# Stage B — Models

In [ ]:
stage("B. fragment-GIN")
frag_rows = []
for mode in MODES:
    gdir, cdir, tag = PATHS[mode]
    for name in DATASETS:
        task = TASKS[name][1]
        _, _, test = load_graphs(name, gdir)
        tests = []
        for seed in SEEDS:
            ck = f"{cdir}/{name}_gin_seed{seed}.pt"
            if RETRAIN or not os.path.exists(ck):
                train_one(name, task, seed, DEVICE, graph_dir=gdir, ckpt_dir=cdir)
            model, c = load_checkpoint(ck, DEVICE)
            tests.append(evaluate(model, DataLoader(test, batch_size=256), task, c["mu"], c["sd"], DEVICE))
        frag_rows.append({"dataset": name, "mode": MODE_NAME[mode], "model": "Fragment-GIN", "metric": METRIC[name],
                          "test_mean": np.mean(tests), "test_std": np.std(tests)})
        print(f"[{MODE_NAME[mode]}] {name} fragment-GIN {METRIC[name]} {np.mean(tests):.3f} ± {np.std(tests):.3f}")
    pd.DataFrame([r for r in frag_rows if r["mode"] == MODE_NAME[mode]])[["dataset", "metric", "test_mean", "test_std"]] \
      .to_csv("gin_results_ions.csv" if mode == "ions" else "gin_results.csv", index=False)

In [ ]:
stage("B. atom-GIN")
atom_rows, atom_specs = [], {m: {} for m in MODES}
for mode in MODES:
    gdir, cdir, tag = PATHS[mode]
    for name in DATASETS:
        task = TASKS[name][1]
        splits = None
        tests = []
        for seed in SEEDS:
            ck = f"{cdir}/{name}_atomgin_seed{seed}.pt"
            if RETRAIN or not os.path.exists(ck):
                splits = splits or build_atom_splits(load_graphs(name, gdir))
                _, test, model, mu, sd = train_atom_gin(name, task, splits, seed, DEVICE, ckpt_dir=cdir)
            else:
                model, mu, sd, _, test = load_atom_gin(ck, DEVICE)
                if test is None:   # older checkpoint without a stored metric
                    splits = splits or build_atom_splits(load_graphs(name, gdir))
                    test = evaluate(model, DataLoader(splits[2], batch_size=256), task, mu, sd, DEVICE)
            tests.append(test)
            if seed == GIN_SEED:
                thr = 0.5 if task == "classification" else float(torch.cat([d.y for d in load_graphs(name, gdir)[0]]).median())
                atom_specs[mode][name] = ModelSpec(model, task, mu, sd, thr)
        atom_rows.append({"dataset": name, "mode": MODE_NAME[mode], "model": "Atom-GIN", "metric": METRIC[name],
                          "test_mean": np.mean(tests), "test_std": np.std(tests)})
        print(f"[{MODE_NAME[mode]}] {name} atom-GIN {METRIC[name]} {np.mean(tests):.3f} ± {np.std(tests):.3f}")

base = pd.DataFrame(frag_rows + atom_rows).sort_values(["dataset", "model", "mode"])
base.to_csv("results/table2_base_models.csv", index=False)
display(base.round(3))
LATEX["Table 2: base models (Dataset & Model & Mode & Metric & Test mean ± std over seeds)"] = [
    f"{r.dataset} & {r.model} & {r['mode']} & {r.metric} & {r.test_mean:.3f} $\\pm$ {r.test_std:.3f} \\\\" for _, r in base.iterrows()]

# Stage C — Diagnostics (no LLM)

In [ ]:
specs = {}
for mode in MODES:
    gdir, cdir, tag = PATHS[mode]
    for name in DATASETS:
        specs[mode, name] = load_spec(name, GIN_SEED, DEVICE, graph_dir=gdir, ckpt_dir=cdir)

if RUN_DIAGNOSTICS:
    stage("C. confusion matrices / parity plots")
    diag_rows, preds = [], {}
    for mode in MODES:
        gdir, cdir, tag = PATHS[mode]
        for name in DATASETS:
            spec = specs[mode, name]
            pred = predict_all_with_sizes(spec, load_graphs(name, gdir)[2], DEVICE); preds[mode, name] = pred
            title = f"{name} | {MODE_NAME[mode]} | fragment-GIN | test"
            if TASKS[name][1] == "classification":
                fig, cm, m = confusion_figure(pred, title, f"figures/diagnostics/confusion_{name}{tag}.png")
                diag_rows.append({"dataset": name, "mode": MODE_NAME[mode], **m,
                                  "TN": cm[0, 0], "FP": cm[0, 1], "FN": cm[1, 0], "TP": cm[1, 1]})
            else:
                fig, m = parity_figure(pred, title, f"figures/diagnostics/parity_{name}{tag}.png", threshold=spec.threshold)
                diag_rows.append({"dataset": name, "mode": MODE_NAME[mode], **m})
            display(fig); plt.close(fig)
    diag = pd.DataFrame(diag_rows); diag.to_csv("results/diagnostic_metrics.csv", index=False); display(diag.round(3))

In [ ]:
if RUN_DIAGNOSTICS:
    stage("C. t-SNE / UMAP")
    for (mode, name), pred in preds.items():
        tag = PATHS[mode][2]
        fig = embedding_figure(pred, TASKS[name][1], f"{name} | {MODE_NAME[mode]} | molecule embeddings (test)",
                               f"figures/diagnostics/embeddings_{name}{tag}.png", threshold=specs[mode, name].threshold)
        display(fig); plt.close(fig)
        train_pred = predict_all_with_sizes(specs[mode, name], load_graphs(name, PATHS[mode][0])[0], DEVICE)
        fig = fragment_embedding_figure(train_pred, f"{name} | {MODE_NAME[mode]} | fragment embeddings (train)",
                                        f"figures/diagnostics/fragment_embeddings_{name}{tag}.png")
        display(fig); plt.close(fig)

In [ ]:
if RUN_DIAGNOSTICS:
    stage("C. readout attention")
    attn_tables = []
    for (mode, name), pred in preds.items():
        tag = PATHS[mode][2]
        fig, st = attention_by_group(pred, f"{name} | {MODE_NAME[mode]} | readout attention by group (test)",
                                     f"figures/diagnostics/attention_by_group_{name}{tag}.png", min_count=5)
        attn_tables.append(st.assign(dataset=name, mode=MODE_NAME[mode]).reset_index()); display(fig); plt.close(fig)
        grid = attention_maps(specs[mode, name], load_graphs(name, PATHS[mode][0])[2], DEVICE, n=6,
                              out_path=f"figures/diagnostics/attention_maps_{name}{tag}.png")
        print(f"{name} | {MODE_NAME[mode]}"); display(grid)
    pd.concat(attn_tables).to_csv("results/attention_by_group.csv", index=False)

# Stage D — Explanations (LLM)

In [ ]:
# Which molecules to explain: same test positions in every mode when PAIRED.
test_sets = {(m, n): load_graphs(n, PATHS[m][0])[2] for m in MODES for n in DATASETS}
indices = {}
for name in DATASETS:
    if PAIRED:
        req = "ions" if (ONLY_SALTS and "ions" in MODES) else None
        indices[name] = paired_indices({m: test_sets[m, name] for m in MODES}, N_EVAL, seed=0, require_counterion=req)
    else:
        indices[name] = None
    print(name, "->", "independent per mode" if indices[name] is None else f"{len(indices[name])} shared molecules")

In [ ]:
records = {m: {} for m in MODES}
if RUN_EXPLAIN:
    stage("D. loading Llama")
    from google.colab import userdata
    try:
        os.environ["HF_TOKEN"] = userdata.get("HF_TOKEN")
    except Exception as e:
        raise RuntimeError("Add a Colab secret named HF_TOKEN (key icon, notebook access on), or set RUN_EXPLAIN = False") from e
    from huggingface_hub import login; login(token=os.environ["HF_TOKEN"])
    tok, llm = load_llm(LLM_ID)
    emb_w = llm.get_input_embeddings().weight
    LLM_DIM, TOKEN_NORM = emb_w.shape[1], emb_w.float().norm(dim=1).mean().item()

    def get_projector(mode, name):
        gdir, cdir, tag = PATHS[mode]
        spec = specs[mode, name]
        path = f"{cdir}/{name}_projector_k{K_TOKENS}_{LV}.pt"
        proj = Projector(spec.gin.head[0].in_features, LLM_DIM, k=K_TOKENS, token_norm=TOKEN_NORM).to(DEVICE)
        if os.path.exists(path):
            proj.load_state_dict(torch.load(path)); proj.eval(); return proj
        Hn, Tn, Hg, Tg = collect_embeddings(spec.gin, load_graphs(name, gdir)[0], DEVICE)
        proj, hist = train_projector(torch.cat([Hn, Hg]), text_embeddings(Tn + Tg, tok, llm), LLM_DIM, TOKEN_NORM,
                                     DEVICE, k=K_TOKENS, epochs=30)
        torch.save(proj.state_dict(), path); print(f"  projector loss {hist[0]:.3f} -> {hist[-1]:.3f}")
        return proj

    for mode in MODES:
        gdir, cdir, tag = PATHS[mode]
        for name in DATASETS:
            stage(f"D. explaining {name} | {MODE_NAME[mode]}")
            extra = {"frag_gradcam": frag_gradcam, "attention": attention,
                     "atom_gradcam": make_atom_gradcam(atom_specs[mode][name])}
            rec, _ = run_explanations(
                specs[mode, name], get_projector(mode, name), tok, llm, test_sets[mode, name], name, DEVICE,
                n_eval=N_EVAL, text_only_baseline=TEXT_ONLY, extra_explainers=extra,
                records_path=f"results/explain_{name}{tag}_{LV}.jsonl", log_every=10,
                only_counterions=(ONLY_SALTS and mode == "ions"), indices=indices[name])
            records[mode][name] = rec

# Stage E — Tables

Uses the explanations from this session, or reloads them from `results/explain_*.jsonl` (so with `RUN_EXPLAIN = False` you can rebuild tables and figures without the LLM).

In [ ]:
def load_records(mode, name):
    path = f"results/explain_{name}{PATHS[mode][2]}_{LV}.jsonl"
    if not os.path.exists(path):
        return pd.DataFrame()
    pos = {g.smiles: i for i, g in enumerate(test_sets[mode, name])}
    rows = [json.loads(l) for l in open(path) if l.strip()]
    for r in rows:
        r.setdefault("test_index", pos.get(r["smiles"]))
        r.setdefault("counterion", [False] * r["n_fragments"])
    df = pd.DataFrame(rows)
    if indices.get(name) is not None and len(df):
        df = df[df["test_index"].isin(set(indices[name]))]
    return df.reset_index(drop=True)

for mode in MODES:
    for name in DATASETS:
        if name not in records[mode] or len(records[mode][name]) == 0:
            records[mode][name] = load_records(mode, name)
EXTRA_KEYS = {"frag_gradcam": 1, "attention": 1, "atom_gradcam": 1}
have = {(m, n): len(records[m][n]) for m in MODES for n in DATASETS}
print("explained molecules:", have)

summ = []
for mode in MODES:
    for name in DATASETS:
        r = records[mode][name]
        if len(r):
            s = summarize(r, EXTRA_KEYS); s.insert(0, "mode", MODE_NAME[mode]); s.insert(0, "dataset", name); summ.append(s)
if summ:
    summary_all = pd.concat(summ); summary_all.to_csv("results/table3_explanations_by_mode.csv", index=False)
    fid = summary_all.pivot_table(index=["dataset", "method"], columns="mode", values="fidelity_%", sort=False)
    size = summary_all.pivot_table(index=["dataset", "method"], columns="mode", values="size", sort=False)
    side = fid.add_suffix(" | fidelity %").join(size.add_suffix(" | size"))
    side.to_csv("results/table3_side_by_side.csv"); display(side.round(2))
    cols = [MODE_NAME[m] for m in MODES if MODE_NAME[m] in fid.columns]
    LATEX["Table 3: explanations (Dataset & Method & fidelity % per mode & size per mode), modes: " + ", ".join(cols)] = [
        f"{n} & {meth} & " + " & ".join(f"{r[c + ' | fidelity %']:.1f}" for c in cols) + " & "
        + " & ".join(f"{r[c + ' | size']:.2f}" for c in cols) + " \\\\" for (n, meth), r in side.iterrows()]

In [ ]:
agree, usage = [], []
for mode in MODES:
    for name in DATASETS:
        r = records[mode][name]
        if not len(r): continue
        a = agreement_table(r); a.insert(0, "mode", MODE_NAME[mode]); a.insert(0, "dataset", name); agree.append(a)
        if mode == "ions":
            u = counterion_usage(r)
            if len(u): u.insert(0, "dataset", name); usage.append(u)
if agree:
    agree = pd.concat(agree); agree.to_csv("results/agreement_by_mode.csv", index=False)
    print("Agreement with GSPELL's supporting fragments:"); display(agree.round(3))
if usage:
    usage = pd.concat(usage); usage.to_csv("results/counterion_usage.csv", index=False)
    print("Counter-ion usage (ions mode):"); display(usage.round(1))

# Stage E2 — Faithfulness: Shapley, counterfactuals, QA

The project's own contribution. For every explained molecule:
1. **Exact fragment Shapley values** (Roth, JCIM 2026): all 2^n fragment subsets are run through the GIN (n ≤ 12), so each fragment's contribution is exact. The value of a subset is the GIN's output on that subgraph, oriented toward the predicted class (positive = supports it); the empty set is the head applied to an empty molecule. Checked with the efficiency axiom.
2. **Removal counterfactuals:** whether removing each fragment alone flips the class (read from the same table).
3. **Agreement:** how well GSPELL, the text-only LLM, Grad-CAM, attention and GNNExplainer match those references, and **deletion curves**: remove fragments in each method's ranked order and count how many go before the class flips (lower = the method found the decisive fragments; Shapley is the reference, random order is chance).
4. **QA faithfulness** (needs the LLM): questions whose answers come from the model (most important fragment, most opposing fragment, would removing fragment k flip the prediction, which of two fragments matters more), asked with and without the GIN's soft prompts. If accuracy is no higher with soft prompts, the rationale reflects the LLM's chemistry knowledge, not the model.

No LLM is needed for 1–3, and results are cached per molecule in `results/faithfulness_*.jsonl`.

In [ ]:
stage("E2. exact Shapley, removal counterfactuals, agreement")
faith = {m: {} for m in MODES}
for mode in MODES:
    gdir, cdir, tag = PATHS[mode]
    for name in DATASETS:
        rec = records[mode][name]
        if not len(rec): continue
        path = f"results/faithfulness_{name}{tag}_{LV}_seed{GIN_SEED}.jsonl"
        done = {r["smiles"]: r for r in load_jsonl(path)}
        rows, t0 = [], time.time()
        for _, r in rec.iterrows():
            if r["smiles"] in done:
                rows.append(done[r["smiles"]]); continue
            data = canonical_graph(test_sets[mode, name][int(r["test_index"])].cpu())
            out = analyse_molecule(specs[mode, name], data, DEVICE, rec=r.to_dict())
            out["test_index"] = int(r["test_index"])
            append_jsonl(path, out); rows.append(out)
        faith[mode][name] = rows
        err = max(x["efficiency_error"] for x in rows)
        flips = np.mean([any(x["removal_flips"]) for x in rows])
        print(f"[{MODE_NAME[mode]}] {name}: {len(rows)} molecules, max efficiency error {err:.1e}, "
              f"{100 * flips:.0f}% have a fragment whose removal alone flips the class ({time.time() - t0:.0f}s)")
        # attach Shapley values and flips to the explanation records for the figures
        by = {x["smiles"]: x for x in rows}
        records[mode][name]["shapley_scores"] = pd.Series([by[s_]["shapley_scores"] for s_ in rec["smiles"]], index=rec.index, dtype=object)
        records[mode][name]["removal_flips"] = pd.Series([by[s_]["removal_flips"] for s_ in rec["smiles"]], index=rec.index, dtype=object)

In [ ]:
agree_tabs, del_tabs, t4 = {m: {} for m in MODES}, {m: {} for m in MODES}, []
for mode in MODES:
    for name, rows in faith[mode].items():
        a = summarize_agreement(rows); d = summarize_deletion(rows)
        agree_tabs[mode][name], del_tabs[mode][name] = a, d
        t4.append(d.assign(dataset=name, mode=MODE_NAME[mode]).reset_index().rename(columns={"index": "method"}))
        print(f"\n[{MODE_NAME[mode]}] {name}: deletion (lower steps_to_flip = finds decisive fragments sooner)")
        display(d.round(3))
        print("agreement with Shapley and removal counterfactuals")
        display(a.round(3))
if t4:
    t4 = pd.concat(t4); t4.to_csv("results/table4_deletion.csv", index=False)
    pd.concat([a.assign(dataset=n, mode=MODE_NAME[m]) for m in MODES for n, a in agree_tabs[m].items()]) \
      .to_csv("results/table4_agreement.csv")
    LATEX["Table 4: share of fragments removed before the class flips (Dataset & Mode & Method & steps & deletion AUC)"] = [
        f"{r.dataset} & {r['mode']} & {r.method} & {r.steps_to_flip:.3f} & {r.deletion_auc:.3f} \\\\" for _, r in t4.iterrows()]
    rows_ = []
    for m in MODES:
        for n, a in agree_tabs[m].items():
            for meth in ["GSPELL (graph-level)", "LLM text-only (no projector)"]:
                if meth in a.index:
                    x = a.loc[meth]
                    rows_.append(f"{n} & {MODE_NAME[m]} & {meth} & {x.get('auroc_phi', np.nan):.3f} & "
                                 f"{x.get('precision_support', np.nan):.3f} & {x.get('recall_decisive', np.nan):.3f} \\\\")
    LATEX["Table 4b: LLM support vs. Shapley (Dataset & Mode & Method & AUROC & precision of YES & recall of decisive)"] = rows_

In [ ]:
qa_tabs = {m: {} for m in MODES}
llm_ready = "llm" in globals() and RUN_EXPLAIN
if RUN_QA and not llm_ready:
    print("QA needs the LLM (RUN_EXPLAIN = True). Showing any saved QA results instead.")
for mode in MODES:
    gdir, cdir, tag = PATHS[mode]
    for name, rows in faith[mode].items():
        path = f"results/qa_{name}{tag}_{LV}_seed{GIN_SEED}.jsonl"
        if RUN_QA and llm_ready:
            stage(f"E2. QA faithfulness {name} | {MODE_NAME[mode]}")
            done = {q["smiles"] for q in load_jsonl(path)}
            spec, proj = specs[mode, name], get_projector(mode, name)
            for fr in rows:
                if fr["smiles"] in done: continue
                data = canonical_graph(test_sets[mode, name][fr["test_index"]].cpu())
                full = predict(spec, data, DEVICE)
                label = TASK_TEXT[name][0] if full >= spec.threshold else TASK_TEXT[name][1]
                for q in ask_questions(spec, proj, tok, llm, data, label, DEVICE,
                                       np.array(fr["shapley_scores"]), fr["removal_flips"], seed=fr["test_index"]):
                    q.update(smiles=fr["smiles"], test_index=fr["test_index"]); append_jsonl(path, q)
        wanted = {fr["smiles"] for fr in rows}
        qa = [q for q in load_jsonl(path) if q["smiles"] in wanted]
        if qa:
            qa_tabs[mode][name] = summarize_qa(qa)
            print(f"\n[{MODE_NAME[mode]}] {name}: QA accuracy ({len(wanted)} molecules)"); display(qa_tabs[mode][name].round(3))
rows_ = [f"{n} & {MODE_NAME[m]} & {qtype} & {int(r.questions)} & {r.with_soft_prompts:.3f} & {r.text_only:.3f} & {r.chance:.3f} \\\\"
         for m in MODES for n, t in qa_tabs[m].items() for qtype, r in t.iterrows()]
if rows_:
    LATEX["Table 5: QA faithfulness (Dataset & Mode & Question & n & with soft prompts & text only & chance)"] = rows_
    pd.concat([t.assign(dataset=n, mode=MODE_NAME[m]) for m in MODES for n, t in qa_tabs[m].items()]).to_csv("results/table5_qa.csv")

In [ ]:
for mode in MODES:
    tag = PATHS[mode][2]
    if del_tabs[mode]:
        fig = faithfulness_figure(del_tabs[mode], agree_tabs[mode], f"Explanations vs. exact Shapley | {MODE_NAME[mode]}",
                                  f"figures/shapley_agreement{tag}.png")
        display(fig); plt.close(fig)
    if qa_tabs[mode]:
        fig = qa_figure(qa_tabs[mode], f"QA faithfulness | {MODE_NAME[mode]}", f"figures/qa_faithfulness{tag}.png")
        display(fig); plt.close(fig)
print("The covalent-only versions (no suffix) match the placeholder figure names in the report.")

# Stage F — Explanation figures

In [ ]:
def figure_molecules(name, n=N_FIG):
    if "ions" in MODES and len(records["ions"][name]):
        r = records["ions"][name]
        salts = r[r["counterion"].apply(any)]["test_index"].tolist()
        return (salts + [i for i in r["test_index"] if i not in salts])[:n]
    first = next((records[m][name] for m in MODES if len(records[m][name])), pd.DataFrame())
    return first["test_index"].tolist()[:n] if len(first) else []

if RUN_FIGURES:
    stage("F. explanation grids and fragment panels")
    for name in DATASETS:
        picks = figure_molecules(name)
        for mode in MODES:
            rec = records[mode][name]
            if not len(rec) or not picks: continue
            rec = rec.set_index("test_index").loc[[i for i in picks if i in set(rec["test_index"])]].reset_index()
            tag = PATHS[mode][2]
            grid = explanation_grid(rec, atom_specs[mode][name], DEVICE, n=len(rec), out_path=f"figures/cam_examples_{name}{tag}.png")
            panel = fragment_panels(rec, n=len(rec), color_by=COLOR_BY, out_path=f"figures/fragment_panels_{name}{tag}.png")
            print(f"{name} | {MODE_NAME[mode]}"); display(grid); display(panel)

In [ ]:
if RUN_FIGURES:
    stage("F. explanation heatmaps")
    for name in DATASETS:
        for i in figure_molecules(name, 2):
            for mode in MODES:
                rec = records[mode][name]
                if not len(rec): continue
                rows = rec[rec["test_index"] == i]
                if not len(rows): continue
                r = rows.iloc[0]
                fig = explanation_heatmap(r, f"figures/heatmap_{name}_{i}{PATHS[mode][2]}.png",
                                          f"{name} #{i} | {MODE_NAME[mode]} | GIN score {r['gin_score']:.2f}")
                display(fig); plt.close(fig)

In [ ]:
if RUN_FIGURES and "llm" in globals():
    stage("F. soft-prompt alignment")
    align = []
    for mode in MODES:
        for name in DATASETS:
            proj = get_projector(mode, name)
            fig, cos = alignment_figure(specs[mode, name], proj, test_sets[mode, name], lambda t: text_embeddings(t, tok, llm),
                                        DEVICE, f"{name} | {MODE_NAME[mode]}", f"figures/alignment_{name}{PATHS[mode][2]}.png")
            align.append({"dataset": name, "mode": MODE_NAME[mode], "mean_cosine": cos}); display(fig); plt.close(fig)
    pd.DataFrame(align).to_csv("results/softprompt_alignment.csv", index=False)
elif RUN_FIGURES:
    print("Soft-prompt alignment needs the LLM (RUN_EXPLAIN = True); skipped.")

# Stage G — Export

In [ ]:
stage("G. export")
with open("results/latex_rows.txt", "w") as f:
    for section, rows in LATEX.items():
        f.write(f"% {section}\n" + "\n".join(rows) + "\n\n")
print(open("results/latex_rows.txt").read())

import shutil
stamp = time.strftime("%Y%m%d_%H%M")
os.makedirs("exports", exist_ok=True)
tmp = f"/content/export_{stamp}"
shutil.rmtree(tmp, ignore_errors=True)
shutil.copytree("figures", f"{tmp}/figures"); shutil.copytree("results", f"{tmp}/results")
archive = shutil.make_archive(f"exports/export_{stamp}", "zip", tmp)
print(f"\nSaved {archive} (figures + results). Total time {(time.time() - T0) / 60:.1f} min.")

In [ ]:
if DISCONNECT_WHEN_DONE:
    from google.colab import runtime
    runtime.unassign()   # releases the GPU; everything above is already on Drive